In [ ]:
#always run this first!!
#essential reticulate functions that allow us to use python packages in R
#you'll need a conda env with 'leidenalg' and 'pandas' installed to do this
#then route reticulate to the python installed in that conda env with the below functions
Sys.setenv(RETICULATE_PYTHON="/home/welison/.conda/envs/decontX_reticulate/bin/python")
library(reticulate)
reticulate::use_python("/home/welison/.conda/envs/decontX_reticulate/bin/python")
reticulate::use_condaenv("/home/welison/.conda/envs/decontX_reticulate")
reticulate::py_module_available(module='leidenalg') #needs to be TRUE
reticulate::import('leidenalg') #good to make sure this doesn't error

In [ ]:
#is you get messages that any of these packages aren't installed
#go ahead and locally install them yourself using install.packages()

#library(GenomeInfoDb,lib.loc="/nfs/lab/welison/multiome_practice/lib/")

suppressMessages(library(hdf5r))
suppressMessages(library(Seurat))
suppressMessages(library(Signac))
suppressMessages(library(EnsDb.Hsapiens.v86))
suppressMessages(library(dplyr))
suppressMessages(library(ggplot2))
suppressMessages(library(Matrix))
#install.packages('harmony')
suppressMessages(library(harmony))
suppressMessages(library(data.table))
#suppressMessages(library(ggpubr))
library(gridExtra)
warnLevel <- getOption('warn')
options(warn = -1)
library(BPCells)
library(stringr)
library(tidyr)
library(ggh4x) # Luca's Dotplot code
library(presto)

In [ ]:
library(RhpcBLASctl)
RhpcBLASctl::blas_set_num_threads(24)

In [ ]:
sessionInfo()

In [ ]:
#This is the marker gene list that has 3 each and the additional cell types we discovered last time. 
# reLoad marker list that has additional cell types and compartment 
#CHANGE file name 
# paste means con
cell.markers = read.table("/nfs/lab/scorban/fnih_liver_231110/intregration/assests/Liver.Marker.Gene.List.3only_2.txt", sep = "\t", header = TRUE)
# Make it long, remove useless column and void markers
cell.markers <- cell.markers %>% gather(Key, marker, c(3:ncol(cell.markers)))
cell.markers = cell.markers[,-3]
cell.markers = cell.markers[cell.markers$marker != "", ]


# Factorize columns -- creating an order of the cells for the graphs 
cell.markers$Compartment = factor(cell.markers$Compartment, 
                        levels = c("Liver", "Immune", "Vascular", "Stromal", "Other"))
cell.markers$CellType = factor(cell.markers$CellType,
                        levels = c("Hepatoblasts", "Hepatocyte", "Cholangiocyte", "Hepatic_stellate_cell", "Kupffer_cell", 
                                   "Macrophage", "Myeloid", "B_cell", "Mast", "DC", "Plasma", "T_cell-NK", "T_cell", "NK", 
                                   "Endothelial", "Lymph-Endo", 
                                   "Fibroblast", "Adipocyte", "Myofibroblast", 
                                   "Erythrocytes"))

cell.compartment = cell.markers[,-3]

In [ ]:
adata_sub <- readRDS('/nfs/lab/projects/nash_nafld_liver/keep_demux/subclustering_share/240711_fnih_liver_ALL28lanes_ALL87donors_subclustering_BPCells.rds')
adata_sub

In [ ]:
#CHANGE NOTHING
#Making UMAPs that are colored by cluster, will need to assign cell types later. 
options(repr.plot.width=18, repr.plot.height=6)
p1 <- DimPlot(adata_sub, reduction='umap.rna', group.by='seurat_clusters', label=TRUE, label.size=6, repel=TRUE) + ggtitle('RNA')
p1 <- p1 + xlab('UMAP 1') + ylab('UMAP 2') + ggtitle('RNA only')
p2 <- DimPlot(adata_sub, reduction='umap.atac', group.by='seurat_clusters', label=TRUE, label.size=6, repel=TRUE) + ggtitle('ATAC')
p2 <- p2 + xlab('UMAP 1') + ylab('UMAP 2') + ggtitle('ATAC only')
p3 <- DimPlot(adata_sub, reduction='umap.wnn', group.by='seurat_clusters', label=TRUE, label.size=6, repel=TRUE) + ggtitle('WNN')
p3 <- p3 + xlab('UMAP 1') + ylab('UMAP 2') + ggtitle('Combined')
p1 + p2 + p3 & NoLegend() & theme(plot.title=element_text(hjust=0.5))

In [ ]:
#CHANGE NOTHING
#Making UMAPs that are colored by cluster, will need to assign cell types later. 
options(repr.plot.width=18, repr.plot.height=6)
p1 <- DimPlot(adata_sub, reduction='umap.rna', group.by='seurat_clusters', label=TRUE, label.size=6, repel=TRUE) + ggtitle('RNA')
p1 <- p1 + xlab('UMAP 1') + ylab('UMAP 2') + ggtitle('RNA only')
p2 <- DimPlot(adata_sub, reduction='umap.atac', group.by='seurat_clusters', label=TRUE, label.size=6, repel=TRUE) + ggtitle('ATAC')
p2 <- p2 + xlab('UMAP 1') + ylab('UMAP 2') + ggtitle('ATAC only')
p3 <- DimPlot(adata_sub, reduction='umap.wnn', group.by='seurat_clusters', label=TRUE, label.size=6, repel=TRUE) + ggtitle('WNN')
p3 <- p3 + xlab('UMAP 1') + ylab('UMAP 2') + ggtitle('Combined')
p1 + p2 + p3 & NoLegend() & theme(plot.title=element_text(hjust=0.5))

In [ ]:
options(repr.plot.width=40, repr.plot.height=15)
#p1 <- DimPlot(adata, reduction='umap.wnn', group.by='donor_demux', split.by='donor_demux', label=FALSE, label.size=10, repel=TRUE)
adata_sub$value <- 1
p2 <- ggplot(adata_sub[[]], aes(fill=donor_demux, y=value, x=seurat_clusters)) + geom_bar(position=position_fill(reverse=TRUE), stat='identity') + xlab('') + ylab('percentage') + theme_light()
p2

In [ ]:
options(repr.plot.width=40, repr.plot.height=15)
#p1 <- DimPlot(adata, reduction='umap.wnn', group.by='donor_demux', split.by='donor_demux', label=FALSE, label.size=10, repel=TRUE)
adata_sub$value <- 1
p2 <- ggplot(adata_sub[[]], aes(fill=condition, y=value, x=seurat_clusters)) + geom_bar(position=position_fill(reverse=TRUE), stat='identity') + xlab('') + ylab('percentage') + theme_light()
p2

In [ ]:
options(repr.plot.width=40, repr.plot.height=15)
#p1 <- DimPlot(adata, reduction='umap.wnn', group.by='donor_demux', split.by='donor_demux', label=FALSE, label.size=10, repel=TRUE)
adata_sub$value <- 1
p2 <- ggplot(adata_sub[[]], aes(fill=batch, y=value, x=seurat_clusters)) + geom_bar(position=position_fill(reverse=TRUE), stat='identity') + xlab('') + ylab('percentage') + theme_light()
p2

In [ ]:
options(repr.plot.width=40, repr.plot.height=15)
#p1 <- DimPlot(adata, reduction='umap.wnn', group.by='donor_demux', split.by='donor_demux', label=FALSE, label.size=10, repel=TRUE)
adata_sub$value <- 1
p2 <- ggplot(adata_sub[[]], aes(fill=lane, y=value, x=seurat_clusters)) + geom_bar(position=position_fill(reverse=TRUE), stat='identity') + xlab('') + ylab('percentage') + theme_light()
p2

In [ ]:
options(repr.plot.width=40, repr.plot.height=15)
#p1 <- DimPlot(adata, reduction='umap.wnn', group.by='donor_demux', split.by='donor_demux', label=FALSE, label.size=10, repel=TRUE)
adata_sub$value <- 1
p2 <- ggplot(adata_sub[[]], aes(fill=disease_status, y=value, x=seurat_clusters)) + geom_bar(position=position_fill(reverse=TRUE), stat='identity') + xlab('') + ylab('percentage') + theme_light()
p2

In [ ]:
gc()

In [ ]:
#reMaking a dot plot with cell type markers. We will use this to not only assign cell types, but also narrow down the markers, since I have too many liver marker genes.
#Change width and height, since I'll have so many markers to visualize. (previous width/height was 25/10).
g = DotPlot(adata_sub, assay='SCT', features=cell.markers$marker, cluster.idents=TRUE, col.min=0) +
        theme(axis.text.x=element_text(angle=45, hjust=1)) + xlab('') + ylab('')
    meta_summary = g$data
    colnames(meta_summary)[3] = "marker"
    meta_summary = merge(meta_summary, cell.markers, by = "marker")

    options(repr.plot.width=25, repr.plot.height=10)
    figure <- ggplot(meta_summary, aes(x = marker, y = id)) +
      geom_point(aes(size = pct.exp, fill = avg.exp.scaled, stroke=NA),
                 shape = 21) +
      scale_size("% detected", range = c(0, 6)) +
      scale_fill_gradient(low = "lightgray", high = "blue",
                           guide = guide_colorbar(nbin = 200,
                                                  ticks.colour = "black", frame.colour = "black"),
                           name = "Average\nexpression") +
      ylab("Cluster") + xlab("") +
      theme_bw() +
      theme(axis.text = element_text(size = 100),
            axis.text.x = element_text(size = 20, angle = 45, hjust = 1, color = "black"),
            strip.text.x = element_text(size = 14),
            axis.text.y = element_text(size = 20, color = "black"),
            axis.title = element_text(size = 20)) +
      facet_nested(cols = vars(Compartment, CellType), scales = "free", space = "free")
figure

In [ ]:
barcodes.celltype <- data.frame()

# T/NK

In [ ]:
adata_cluster_T <- subset(adata_sub, subset=seurat_clusters %in% as.character(c(13,16)))
adata_cluster_T

In [ ]:
adata_cluster_T <- FindClusters(adata_cluster_T, graph.name='wsnn', algorithm=4, cluster.name='wnn_sub_cluster',
                              resolution = .5, verbose=TRUE, method = 'igraph')

In [ ]:
options(repr.plot.width=12, repr.plot.height=10)
DimPlot(adata_cluster_T, reduction='umap.wnn', label=TRUE, label.size=6, repel=TRUE) +
ggtitle('WNN') + xlab('UMAP 1') + ylab('UMAP 2') + ggtitle('Combined')

In [ ]:
options(repr.plot.width=12, repr.plot.height=10)
DimPlot(adata_cluster_T, reduction='umap.wnn', group.by = 'condition', label=TRUE, label.size=6, repel=TRUE) +
ggtitle('WNN') + xlab('UMAP 1') + ylab('UMAP 2') + ggtitle('Combined')

In [ ]:
options(repr.plot.width=20, repr.plot.height=15)
#p1 <- DimPlot(adata, reduction='umap.wnn', group.by='donor_demux', split.by='donor_demux', label=FALSE, label.size=10, repel=TRUE)
#adata$value <- 1
p2 <- ggplot(adata_cluster_T[[]], aes(fill=condition, y=value, x=seurat_clusters)) + geom_bar(position=position_fill(reverse=TRUE), stat='identity') + xlab('') + ylab('percentage') + theme_light()
p2

In [ ]:
options(repr.plot.width=20, repr.plot.height=15)
#p1 <- DimPlot(adata, reduction='umap.wnn', group.by='donor_demux', split.by='donor_demux', label=FALSE, label.size=10, repel=TRUE)
#adata$value <- 1
p2 <- ggplot(adata_cluster_T[[]], aes(fill=donor_demux, y=value, x=seurat_clusters)) + geom_bar(position=position_fill(reverse=TRUE), stat='identity') + xlab('') + ylab('percentage') + theme_light()
p2

In [ ]:
marker.set <- cell.markers

#reMaking a dot plot with cell type markers. We will use this to not only assign cell types, but also narrow down the markers, since I have too many liver marker genes.
#Change width and height, since I'll have so many markers to visualize. (previous width/height was 25/10).
g = DotPlot(adata_cluster_T, assay='SCT', features=marker.set$marker, group.by='wnn_sub_cluster', col.min=0) +
        theme(axis.text.x=element_text(angle=45, hjust=1)) + xlab('') + ylab('')
    meta_summary = g$data
    colnames(meta_summary)[3] = "marker"
    meta_summary = merge(meta_summary, marker.set, by = "marker")

    options(repr.plot.width=20, repr.plot.height=10)
    figure <- ggplot(meta_summary, aes(x = marker, y = id)) +
      geom_point(aes(size = pct.exp, fill = avg.exp.scaled, stroke=NA),
                 shape = 21) +
      scale_size("% detected", range = c(0, 6)) +
      scale_fill_gradient(low = "lightgray", high = "blue",
                           guide = guide_colorbar(nbin = 200,
                                                  ticks.colour = "black", frame.colour = "black"),
                           name = "Average\nexpression") +
      ylab("Cluster") + xlab("") +
      theme_bw() +
      theme(axis.text = element_text(size = 100),
            axis.text.x = element_text(size = 20, angle = 45, hjust = 1, color = "black"),
            strip.text.x = element_text(size = 14),
            axis.text.y = element_text(size = 20, color = "black"),
            axis.title = element_text(size = 20)) +
      facet_nested(cols = vars(Compartment, CellType), scales = "free", space = "free")
figure

In [ ]:
options(repr.plot.width=10, repr.plot.height=20)
p1 <- VlnPlot(adata_cluster_T, features='nCount_SCT', group.by='wnn_sub_cluster', pt.size=0, log=TRUE) + geom_boxplot(width=.6, fill='white', alpha=.6) + geom_hline(yintercept=median(adata_cluster_T$nCount_SCT), linetype='dashed')
p2 <- VlnPlot(adata_cluster_T, features='nFeature_SCT', group.by='wnn_sub_cluster', pt.size=0, log=TRUE) + geom_boxplot(width=.6, fill='white', alpha=.6) + geom_hline(yintercept=median(adata_cluster_T$nFeature_SCT), linetype='dashed')
p3 <- VlnPlot(adata_cluster_T, features='nCount_ATAC', group.by='wnn_sub_cluster', pt.size=0, log=TRUE) + geom_boxplot(width=.6, fill='white', alpha=.6) + geom_hline(yintercept=median(adata_cluster_T$nCount_ATAC), linetype='dashed')
p4 <- VlnPlot(adata_cluster_T, features='nFeature_ATAC', group.by='wnn_sub_cluster', pt.size=0, log=TRUE) + geom_boxplot(width=.6, fill='white', alpha=.6) + geom_hline(yintercept=median(adata_cluster_T$nFeature_ATAC), linetype='dashed')
p1 / p2 / p3 / p4

In [ ]:
options(repr.plot.width=8, repr.plot.height=8)

t.nk.markers <- c('KLRF1','NCAM1','NCR1','IL2RB','THEMIS','CAMK4','IL7R','INPP4B')

for (mark in t.nk.markers) {
    p1 <- FeaturePlot(
      object = adata_cluster_T,
      reduction = "umap.wnn",
      features = c(mark),
      ncol = 1,
      raster=TRUE,
      order=T
    ) + xlim(c(-15, -10)) + ylim(c(2.5,12.5)) + scale_colour_viridis_c(option='rocket', direction=-1)
    
    print(p1)
}

In [ ]:
TNK.markers <- FindAllMarkers(adata_cluster_T, assay = 'SCT')

dim(TNK.markers)
head(TNK.markers)

In [ ]:
t.nk.de.markers <- FindMarkers(adata_cluster_T, assay = 'SCT', ident.1=c(1,2), ident.2=3:7)
t.nk.de.markers

In [ ]:
rownames(head(filter(t.nk.de.markers, pct.1 > pct.2), n=20))
head(filter(t.nk.de.markers, pct.1 > pct.2), n=20)

In [ ]:
rownames(head(filter(t.nk.de.markers, pct.1 < pct.2), n=20))
head(filter(t.nk.de.markers, pct.1 < pct.2), n=20)

In [ ]:
save_data <- adata_cluster_T
save_data[["RNA"]] <- as(object = save_data[["RNA"]], Class = "Assay")
save_data[["SCT"]] <- as(object = save_data[["SCT"]], Class = "Assay")
save_data[["windows"]] <- as(object = save_data[["windows"]], Class = "Assay")
save_data[["RNA_raw"]] <- as(object = save_data[["RNA_raw"]], Class = "Assay")
saveRDS(save_data, '/nfs/lab/projects/nash_nafld_liver/downstream_all/windows_subclustering/240806_fnih_liver_ALL28lanes_ALL87donors_subclustering_T.NK_cells.rds')
saveRDS(adata_cluster_T, '/nfs/lab/projects/nash_nafld_liver/downstream_all/windows_subclustering/240806_fnih_liver_ALL28lanes_ALL87donors_subclustering_T.NK_cells_BPCells.rds')
save_data <- NULL
gc()

In [ ]:
write.table(t.nk.de.markers, paste0('/nfs/lab/projects/nash_nafld_liver/downstream_all/windows_subclustering/240806_fnih_liver_ALL28lanes_ALL87donors_subclustering_T.NK_cells_cluster_markers.tsv'),
            sep='\t', quote=F, col.names=T, row.names=T)
write.table(TNK.markers, paste0('/nfs/lab/projects/nash_nafld_liver/downstream_all/windows_subclustering/240806_fnih_liver_ALL28lanes_ALL87donors_subclustering_T.NK_cells_T_vs_NK_markers.tsv'),
            sep='\t', quote=F, col.names=T, row.names=T)

In [ ]:
# NK
barcodes.celltype.tmp = data.frame(barcode = WhichCells(adata_cluster_T, ident = c(1:2)),
                                   celltype = "NK")
barcodes.celltype = rbind(barcodes.celltype, barcodes.celltype.tmp)

# T
barcodes.celltype.tmp = data.frame(barcode = WhichCells(adata_cluster_T, ident = c(3:7)),
                                   celltype = "T")
barcodes.celltype = rbind(barcodes.celltype, barcodes.celltype.tmp)

# Endothelial

In [ ]:
adata_cluster_endo <- subset(adata_sub, subset=seurat_clusters %in% as.character(c(2,23)))
adata_cluster_endo

In [ ]:
adata_cluster_endo <- FindClusters(adata_cluster_endo, graph.name='wsnn', algorithm=4, cluster.name='wnn_sub_cluster',
                              resolution = .5, verbose=TRUE, method = 'igraph')

In [ ]:
options(repr.plot.width=12, repr.plot.height=10)
DimPlot(adata_cluster_endo, reduction='umap.wnn', label=TRUE, label.size=6, repel=TRUE) +
ggtitle('WNN') + xlab('UMAP 1') + ylab('UMAP 2') + ggtitle('Combined')

In [ ]:
options(repr.plot.width=12, repr.plot.height=10)
DimPlot(adata_cluster_endo, reduction='umap.wnn', group.by = 'condition', label=TRUE, label.size=6, repel=TRUE) +
ggtitle('WNN') + xlab('UMAP 1') + ylab('UMAP 2') + ggtitle('Combined')

In [ ]:
options(repr.plot.width=20, repr.plot.height=15)
#p1 <- DimPlot(adata, reduction='umap.wnn', group.by='donor_demux', split.by='donor_demux', label=FALSE, label.size=10, repel=TRUE)
#adata$value <- 1
p2 <- ggplot(adata_cluster_endo[[]], aes(fill=condition, y=value, x=seurat_clusters)) + geom_bar(position=position_fill(reverse=TRUE), stat='identity') + xlab('') + ylab('percentage') + theme_light()
p2

In [ ]:
options(repr.plot.width=20, repr.plot.height=15)
#p1 <- DimPlot(adata, reduction='umap.wnn', group.by='donor_demux', split.by='donor_demux', label=FALSE, label.size=10, repel=TRUE)
#adata$value <- 1
p2 <- ggplot(adata_cluster_endo[[]], aes(fill=donor_demux, y=value, x=seurat_clusters)) + geom_bar(position=position_fill(reverse=TRUE), stat='identity') + xlab('') + ylab('percentage') + theme_light()
p2

In [ ]:
marker.set <- cell.markers

#reMaking a dot plot with cell type markers. We will use this to not only assign cell types, but also narrow down the markers, since I have too many liver marker genes.
#Change width and height, since I'll have so many markers to visualize. (previous width/height was 25/10).
g = DotPlot(adata_cluster_endo, assay='SCT', features=marker.set$marker, group.by='wnn_sub_cluster', col.min=0) +
        theme(axis.text.x=element_text(angle=45, hjust=1)) + xlab('') + ylab('')
    meta_summary = g$data
    colnames(meta_summary)[3] = "marker"
    meta_summary = merge(meta_summary, marker.set, by = "marker")

    options(repr.plot.width=20, repr.plot.height=10)
    figure <- ggplot(meta_summary, aes(x = marker, y = id)) +
      geom_point(aes(size = pct.exp, fill = avg.exp.scaled, stroke=NA),
                 shape = 21) +
      scale_size("% detected", range = c(0, 6)) +
      scale_fill_gradient(low = "lightgray", high = "blue",
                           guide = guide_colorbar(nbin = 200,
                                                  ticks.colour = "black", frame.colour = "black"),
                           name = "Average\nexpression") +
      ylab("Cluster") + xlab("") +
      theme_bw() +
      theme(axis.text = element_text(size = 100),
            axis.text.x = element_text(size = 20, angle = 45, hjust = 1, color = "black"),
            strip.text.x = element_text(size = 14),
            axis.text.y = element_text(size = 20, color = "black"),
            axis.title = element_text(size = 20)) +
      facet_nested(cols = vars(Compartment, CellType), scales = "free", space = "free")
figure

In [ ]:
options(repr.plot.width=10, repr.plot.height=20)
p1 <- VlnPlot(adata_cluster_endo, features='nCount_SCT', group.by='wnn_sub_cluster', pt.size=0, log=TRUE) + geom_boxplot(width=.6, fill='white', alpha=.6) + geom_hline(yintercept=median(adata_cluster_endo$nCount_SCT), linetype='dashed')
p2 <- VlnPlot(adata_cluster_endo, features='nFeature_SCT', group.by='wnn_sub_cluster', pt.size=0, log=TRUE) + geom_boxplot(width=.6, fill='white', alpha=.6) + geom_hline(yintercept=median(adata_cluster_endo$nFeature_SCT), linetype='dashed')
p3 <- VlnPlot(adata_cluster_endo, features='nCount_ATAC', group.by='wnn_sub_cluster', pt.size=0, log=TRUE) + geom_boxplot(width=.6, fill='white', alpha=.6) + geom_hline(yintercept=median(adata_cluster_endo$nCount_ATAC), linetype='dashed')
p4 <- VlnPlot(adata_cluster_endo, features='nFeature_ATAC', group.by='wnn_sub_cluster', pt.size=0, log=TRUE) + geom_boxplot(width=.6, fill='white', alpha=.6) + geom_hline(yintercept=median(adata_cluster_endo$nFeature_ATAC), linetype='dashed')
p1 / p2 / p3 / p4

In [ ]:
Endo.markers <- FindAllMarkers(adata_cluster_endo, assay = 'SCT')

dim(Endo.markers)
head(Endo.markers)

In [ ]:
options(repr.plot.width=8, repr.plot.height=8)

endo.plot.markers <- c('LYVE1','FCGR2B','STAB1','IL7','TBX1','MMRN1','RSPO3','VWF','WNT2','DLL4','EFNB2','JAG1','EDNRB','EFNB1','LTBP4')

for (mark in endo.plot.markers) {
    p1 <- FeaturePlot(
      object = adata_cluster_endo,
      reduction = "umap.wnn",
      features = c(mark),
      ncol = 1,
      raster=TRUE,
      order=T
    ) + xlim(c(-15, 0)) + ylim(c(-15,-5)) + scale_colour_viridis_c(option='rocket', direction=-1)
    
    print(p1)
}

In [ ]:
save_data <- adata_cluster_endo
save_data[["RNA"]] <- as(object = save_data[["RNA"]], Class = "Assay")
save_data[["SCT"]] <- as(object = save_data[["SCT"]], Class = "Assay")
save_data[["windows"]] <- as(object = save_data[["windows"]], Class = "Assay")
save_data[["RNA_raw"]] <- as(object = save_data[["RNA_raw"]], Class = "Assay")
saveRDS(save_data, '/nfs/lab/projects/nash_nafld_liver/downstream_all/windows_subclustering/240806_fnih_liver_ALL28lanes_ALL87donors_subclustering_Endothelial_cells.rds')
saveRDS(adata_cluster_endo, '/nfs/lab/projects/nash_nafld_liver/downstream_all/windows_subclustering/240806_fnih_liver_ALL28lanes_ALL87donors_subclustering_Endothelial_cells_BPCells.rds')
save_data <- NULL
gc()

In [ ]:
saveRDS(adata_cluster_endo, '/nfs/lab/projects/nash_nafld_liver/downstream_all/windows_subclustering/240806_fnih_liver_ALL28lanes_ALL87donors_subclustering_Endothelial_cells_BPCells.rds')

In [ ]:
write.table(Endo.markers, paste0('/nfs/lab/projects/nash_nafld_liver/downstream_all/windows_subclustering/240806_fnih_liver_ALL28lanes_ALL87donors_subclustering_Endothelial_cells_cluster_markers.tsv'),
            sep='\t', quote=F, col.names=T, row.names=T)

In [ ]:
# Sinusoidal.Endothelial
barcodes.celltype.tmp = data.frame(barcode = WhichCells(adata_cluster_endo, ident = c('1','2','3','4','6','7','8','10','11')),
                                   celltype = "Sinusoidal.Endothelial")
barcodes.celltype = rbind(barcodes.celltype, barcodes.celltype.tmp)

# Lymphatic.Endothelial
barcodes.celltype.tmp = data.frame(barcode = WhichCells(adata_cluster_endo, ident = c('12')),
                                   celltype = "Lymphatic.Endothelial")
barcodes.celltype = rbind(barcodes.celltype, barcodes.celltype.tmp)

# Central.Vein.Endothelial
barcodes.celltype.tmp = data.frame(barcode = WhichCells(adata_cluster_endo, ident = c('9')),
                                   celltype = "Central.Vein.Endothelial")
barcodes.celltype = rbind(barcodes.celltype, barcodes.celltype.tmp)

# Vascular.Endothelial
barcodes.celltype.tmp = data.frame(barcode = WhichCells(adata_cluster_endo, ident = c('5')),
                                   celltype = "Vascular.Endothelial")
barcodes.celltype = rbind(barcodes.celltype, barcodes.celltype.tmp)

# Stellate - Revisit

In [ ]:
adata_cluster_stellate <- subset(adata_sub, subset=seurat_clusters %in% as.character(c(11,12,20)))
adata_cluster_stellate

In [ ]:
adata_cluster_stellate <- FindClusters(adata_cluster_stellate, graph.name='wsnn', algorithm=4, cluster.name='wnn_sub_cluster',
                              resolution = .5, verbose=TRUE, method = 'igraph')

In [ ]:
options(repr.plot.width=12, repr.plot.height=10)
DimPlot(adata_cluster_stellate, reduction='umap.wnn', label=TRUE, label.size=6, repel=TRUE) +
ggtitle('WNN') + xlab('UMAP 1') + ylab('UMAP 2') + ggtitle('Combined')

In [ ]:
options(repr.plot.width=12, repr.plot.height=10)
DimPlot(adata_cluster_stellate, reduction='umap.wnn', group.by = 'condition', label=TRUE, label.size=6, repel=TRUE) +
ggtitle('WNN') + xlab('UMAP 1') + ylab('UMAP 2') + ggtitle('Combined')

In [ ]:
options(repr.plot.width=20, repr.plot.height=15)
#p1 <- DimPlot(adata, reduction='umap.wnn', group.by='donor_demux', split.by='donor_demux', label=FALSE, label.size=10, repel=TRUE)
#adata$value <- 1
p2 <- ggplot(adata_cluster_stellate[[]], aes(fill=condition, y=value, x=seurat_clusters)) + geom_bar(position=position_fill(reverse=TRUE), stat='identity') + xlab('') + ylab('percentage') + theme_light()
p2

In [ ]:
options(repr.plot.width=20, repr.plot.height=15)
#p1 <- DimPlot(adata, reduction='umap.wnn', group.by='donor_demux', split.by='donor_demux', label=FALSE, label.size=10, repel=TRUE)
#adata$value <- 1
p2 <- ggplot(adata_cluster_stellate[[]], aes(fill=donor_demux, y=value, x=seurat_clusters)) + geom_bar(position=position_fill(reverse=TRUE), stat='identity') + xlab('') + ylab('percentage') + theme_light()
p2

In [ ]:
marker.set <- cell.markers

#reMaking a dot plot with cell type markers. We will use this to not only assign cell types, but also narrow down the markers, since I have too many liver marker genes.
#Change width and height, since I'll have so many markers to visualize. (previous width/height was 25/10).
g = DotPlot(adata_cluster_stellate, assay='SCT', features=marker.set$marker, group.by='wnn_sub_cluster', col.min=0) +
        theme(axis.text.x=element_text(angle=45, hjust=1)) + xlab('') + ylab('')
    meta_summary = g$data
    colnames(meta_summary)[3] = "marker"
    meta_summary = merge(meta_summary, marker.set, by = "marker")

    options(repr.plot.width=20, repr.plot.height=10)
    figure <- ggplot(meta_summary, aes(x = marker, y = id)) +
      geom_point(aes(size = pct.exp, fill = avg.exp.scaled, stroke=NA),
                 shape = 21) +
      scale_size("% detected", range = c(0, 6)) +
      scale_fill_gradient(low = "lightgray", high = "blue",
                           guide = guide_colorbar(nbin = 200,
                                                  ticks.colour = "black", frame.colour = "black"),
                           name = "Average\nexpression") +
      ylab("Cluster") + xlab("") +
      theme_bw() +
      theme(axis.text = element_text(size = 100),
            axis.text.x = element_text(size = 20, angle = 45, hjust = 1, color = "black"),
            strip.text.x = element_text(size = 14),
            axis.text.y = element_text(size = 20, color = "black"),
            axis.title = element_text(size = 20)) +
      facet_nested(cols = vars(Compartment, CellType), scales = "free", space = "free")
figure

In [ ]:
options(repr.plot.width=10, repr.plot.height=20)
p1 <- VlnPlot(adata_cluster_stellate, features='nCount_SCT', group.by='wnn_sub_cluster', pt.size=0, log=TRUE) + geom_boxplot(width=.6, fill='white', alpha=.6) + geom_hline(yintercept=median(adata_cluster_stellate$nCount_SCT), linetype='dashed')
p2 <- VlnPlot(adata_cluster_stellate, features='nFeature_SCT', group.by='wnn_sub_cluster', pt.size=0, log=TRUE) + geom_boxplot(width=.6, fill='white', alpha=.6) + geom_hline(yintercept=median(adata_cluster_stellate$nFeature_SCT), linetype='dashed')
p3 <- VlnPlot(adata_cluster_stellate, features='nCount_ATAC', group.by='wnn_sub_cluster', pt.size=0, log=TRUE) + geom_boxplot(width=.6, fill='white', alpha=.6) + geom_hline(yintercept=median(adata_cluster_stellate$nCount_ATAC), linetype='dashed')
p4 <- VlnPlot(adata_cluster_stellate, features='nFeature_ATAC', group.by='wnn_sub_cluster', pt.size=0, log=TRUE) + geom_boxplot(width=.6, fill='white', alpha=.6) + geom_hline(yintercept=median(adata_cluster_stellate$nFeature_ATAC), linetype='dashed')
p1 / p2 / p3 / p4

In [ ]:
Stellate.markers <- FindAllMarkers(adata_cluster_stellate, assay = 'SCT')

dim(Stellate.markers)
head(Stellate.markers)

In [ ]:
options(repr.plot.width=8, repr.plot.height=8)

stellate.plot.markers <- c('SPON1','SERPINE1','COL1A1','VIPR1','RELN','MYH11','EBF1')

for (mark in stellate.plot.markers) {
    p1 <- FeaturePlot(
      object = adata_cluster_stellate,
      reduction = "umap.wnn",
      features = c(mark),
      ncol = 1,
      raster=TRUE,
      order=T
    ) + xlim(c(-15,-5)) + ylim(c(-5,5)) + scale_colour_viridis_c(option='rocket', direction=-1)
    
    print(p1)
}

In [ ]:
save_data <- adata_cluster_stellate
save_data[["RNA"]] <- as(object = save_data[["RNA"]], Class = "Assay")
save_data[["SCT"]] <- as(object = save_data[["SCT"]], Class = "Assay")
save_data[["windows"]] <- as(object = save_data[["windows"]], Class = "Assay")
save_data[["RNA_raw"]] <- as(object = save_data[["RNA_raw"]], Class = "Assay")
saveRDS(save_data, '/nfs/lab/projects/nash_nafld_liver/downstream_all/windows_subclustering/240806_fnih_liver_ALL28lanes_ALL87donors_subclustering_Stellate_cells.rds')
saveRDS(adata_cluster_stellate, '/nfs/lab/projects/nash_nafld_liver/downstream_all/windows_subclustering/240806_fnih_liver_ALL28lanes_ALL87donors_subclustering_Stellate_cells_BPCells.rds')
save_data <- NULL
gc()

In [ ]:
write.table(Stellate.markers, paste0('/nfs/lab/projects/nash_nafld_liver/downstream_all/windows_subclustering/240806_fnih_liver_ALL28lanes_ALL87donors_subclustering_Stellate_cells_cluster_markers.tsv'),
            sep='\t', quote=F, col.names=T, row.names=T)

In [ ]:
# Quescent.HSC
barcodes.celltype.tmp = data.frame(barcode = WhichCells(adata_cluster_stellate, ident = c('1','5','6')),
                                   celltype = "Quescent.HSC")
barcodes.celltype = rbind(barcodes.celltype, barcodes.celltype.tmp)

# Activated.HSC
barcodes.celltype.tmp = data.frame(barcode = WhichCells(adata_cluster_stellate, ident = c('2','3')),
                                   celltype = "Activated.HSC")
barcodes.celltype = rbind(barcodes.celltype, barcodes.celltype.tmp)

# Mesenchymal
barcodes.celltype.tmp = data.frame(barcode = WhichCells(adata_cluster_stellate, ident = c('4')),
                                   celltype = "Mesenchymal")
barcodes.celltype = rbind(barcodes.celltype, barcodes.celltype.tmp)

In [ ]:
dim(barcodes.celltype)

# Macrophage

In [ ]:
adata_cluster_macro <- subset(adata_sub, subset=seurat_clusters %in% as.character(c(3,15)))
adata_cluster_macro

In [ ]:
adata_cluster_macro <- FindClusters(adata_cluster_macro, graph.name='wsnn', algorithm=4, cluster.name='wnn_sub_cluster',
                              resolution = .5, verbose=TRUE, method = 'igraph')

In [ ]:
options(repr.plot.width=12, repr.plot.height=10)
DimPlot(adata_cluster_macro, reduction='umap.wnn', label=TRUE, label.size=6, repel=TRUE) +
ggtitle('WNN') + xlab('UMAP 1') + ylab('UMAP 2') + ggtitle('Combined')

In [ ]:
options(repr.plot.width=12, repr.plot.height=10)
DimPlot(adata_cluster_macro, reduction='umap.wnn', label=TRUE, label.size=6, repel=TRUE) +
ggtitle('WNN') + xlab('UMAP 1') + ylab('UMAP 2') + ggtitle('Combined')

In [ ]:
options(repr.plot.width=12, repr.plot.height=10)
DimPlot(adata_cluster_macro, reduction='umap.wnn', group.by = 'condition', label=TRUE, label.size=6, repel=TRUE) +
ggtitle('WNN') + xlab('UMAP 1') + ylab('UMAP 2') + ggtitle('Combined')

In [ ]:
options(repr.plot.width=20, repr.plot.height=15)
#p1 <- DimPlot(adata, reduction='umap.wnn', group.by='donor_demux', split.by='donor_demux', label=FALSE, label.size=10, repel=TRUE)
#adata$value <- 1
p2 <- ggplot(adata_cluster_macro[[]], aes(fill=condition, y=value, x=seurat_clusters)) + geom_bar(position=position_fill(reverse=TRUE), stat='identity') + xlab('') + ylab('percentage') + theme_light()
p2

In [ ]:
options(repr.plot.width=20, repr.plot.height=15)
#p1 <- DimPlot(adata, reduction='umap.wnn', group.by='donor_demux', split.by='donor_demux', label=FALSE, label.size=10, repel=TRUE)
#adata$value <- 1
p2 <- ggplot(adata_cluster_macro[[]], aes(fill=donor_demux, y=value, x=seurat_clusters)) + geom_bar(position=position_fill(reverse=TRUE), stat='identity') + xlab('') + ylab('percentage') + theme_light()
p2

In [ ]:
marker.set <- cell.markers

#reMaking a dot plot with cell type markers. We will use this to not only assign cell types, but also narrow down the markers, since I have too many liver marker genes.
#Change width and height, since I'll have so many markers to visualize. (previous width/height was 25/10).
g = DotPlot(adata_cluster_macro, assay='SCT', features=marker.set$marker, group.by='wnn_sub_cluster', col.min=0) +
        theme(axis.text.x=element_text(angle=45, hjust=1)) + xlab('') + ylab('')
    meta_summary = g$data
    colnames(meta_summary)[3] = "marker"
    meta_summary = merge(meta_summary, marker.set, by = "marker")

    options(repr.plot.width=20, repr.plot.height=10)
    figure <- ggplot(meta_summary, aes(x = marker, y = id)) +
      geom_point(aes(size = pct.exp, fill = avg.exp.scaled, stroke=NA),
                 shape = 21) +
      scale_size("% detected", range = c(0, 6)) +
      scale_fill_gradient(low = "lightgray", high = "blue",
                           guide = guide_colorbar(nbin = 200,
                                                  ticks.colour = "black", frame.colour = "black"),
                           name = "Average\nexpression") +
      ylab("Cluster") + xlab("") +
      theme_bw() +
      theme(axis.text = element_text(size = 100),
            axis.text.x = element_text(size = 20, angle = 45, hjust = 1, color = "black"),
            strip.text.x = element_text(size = 14),
            axis.text.y = element_text(size = 20, color = "black"),
            axis.title = element_text(size = 20)) +
      facet_nested(cols = vars(Compartment, CellType), scales = "free", space = "free")
figure

In [ ]:
options(repr.plot.width=10, repr.plot.height=20)
p1 <- VlnPlot(adata_cluster_macro, features='nCount_SCT', group.by='wnn_sub_cluster', pt.size=0, log=TRUE) + geom_boxplot(width=.6, fill='white', alpha=.6) + geom_hline(yintercept=median(adata_cluster_macro$nCount_SCT), linetype='dashed')
p2 <- VlnPlot(adata_cluster_macro, features='nFeature_SCT', group.by='wnn_sub_cluster', pt.size=0, log=TRUE) + geom_boxplot(width=.6, fill='white', alpha=.6) + geom_hline(yintercept=median(adata_cluster_macro$nFeature_SCT), linetype='dashed')
p3 <- VlnPlot(adata_cluster_macro, features='nCount_ATAC', group.by='wnn_sub_cluster', pt.size=0, log=TRUE) + geom_boxplot(width=.6, fill='white', alpha=.6) + geom_hline(yintercept=median(adata_cluster_macro$nCount_ATAC), linetype='dashed')
p4 <- VlnPlot(adata_cluster_macro, features='nFeature_ATAC', group.by='wnn_sub_cluster', pt.size=0, log=TRUE) + geom_boxplot(width=.6, fill='white', alpha=.6) + geom_hline(yintercept=median(adata_cluster_macro$nFeature_ATAC), linetype='dashed')
p1 / p2 / p3 / p4

In [ ]:
Macro.markers <- FindAllMarkers(adata_cluster_macro, assay = 'SCT')

dim(Macro.markers)
head(Macro.markers)

In [ ]:
options(repr.plot.width=8, repr.plot.height=8)

macro.plot.markers <- c('CD5L','MARCO','TIMD4','MNDA','CD9','IFITM2','SLC11A1','FCN1','MKI67','SPP1','CD9','LGALS3','SPP1','GPNMB','FABP5')

for (mark in macro.plot.markers) {
    p1 <- FeaturePlot(
      object = adata_cluster_macro,
      reduction = "umap.wnn",
      features = c(mark),
      ncol = 1,
      raster=TRUE,
      order=T
    ) + scale_colour_viridis_c(option='rocket', direction=-1)
    
    print(p1)
}

In [ ]:
save_data <- adata_cluster_macro
save_data[["RNA"]] <- as(object = save_data[["RNA"]], Class = "Assay")
save_data[["SCT"]] <- as(object = save_data[["SCT"]], Class = "Assay")
save_data[["windows"]] <- as(object = save_data[["windows"]], Class = "Assay")
save_data[["RNA_raw"]] <- as(object = save_data[["RNA_raw"]], Class = "Assay")
saveRDS(save_data, '/nfs/lab/projects/nash_nafld_liver/downstream_all/windows_subclustering/240806_fnih_liver_ALL28lanes_ALL87donors_subclustering_Macrophage_cells.rds')
saveRDS(adata_cluster_macro, '/nfs/lab/projects/nash_nafld_liver/downstream_all/windows_subclustering/240806_fnih_liver_ALL28lanes_ALL87donors_subclustering_Macrophage_cells_BPCells.rds')
save_data <- NULL
gc()

In [ ]:
write.table(Macro.markers, paste0('/nfs/lab/projects/nash_nafld_liver/downstream_all/windows_subclustering/240806_fnih_liver_ALL28lanes_ALL87donors_subclustering_Macrophage_cells_cluster_markers.tsv'),
            sep='\t', quote=F, col.names=T, row.names=T)

In [ ]:
# Sinusoidal.Endothelial
barcodes.celltype.tmp = data.frame(barcode = WhichCells(adata_cluster_macro, ident = c('1','3','5','6','7','8','9')),
                                   celltype = "Kupffer")
barcodes.celltype = rbind(barcodes.celltype, barcodes.celltype.tmp)

# Lymphatic.Endothelial
barcodes.celltype.tmp = data.frame(barcode = WhichCells(adata_cluster_macro, ident = c('2','10')),
                                   celltype = "Macrophage")
barcodes.celltype = rbind(barcodes.celltype, barcodes.celltype.tmp)

# Central.Vein.Endothelial
barcodes.celltype.tmp = data.frame(barcode = WhichCells(adata_cluster_macro, ident = c('4')),
                                   celltype = "Lipid Associated Macrophage")
barcodes.celltype = rbind(barcodes.celltype, barcodes.celltype.tmp)

In [ ]:
dim(barcodes.celltype)

# B/Plasma

In [ ]:
adata_cluster_B <- subset(adata_sub, subset=seurat_clusters %in% as.character(c(18,24)))
adata_cluster_B

In [ ]:
adata_cluster_B <- FindClusters(adata_cluster_B, graph.name='wsnn', algorithm=4, cluster.name='wnn_sub_cluster',
                              resolution = .2, verbose=TRUE, method = 'igraph')

In [ ]:
options(repr.plot.width=12, repr.plot.height=10)
DimPlot(adata_cluster_B, reduction='umap.wnn', label=TRUE, label.size=6, repel=TRUE) +
ggtitle('WNN') + xlab('UMAP 1') + ylab('UMAP 2') + ggtitle('Combined')

In [ ]:
options(repr.plot.width=12, repr.plot.height=10)
DimPlot(adata_cluster_B, reduction='umap.wnn', label=TRUE, label.size=6, repel=TRUE) +
ggtitle('WNN') + xlab('UMAP 1') + ylab('UMAP 2') + ggtitle('Combined')

In [ ]:
options(repr.plot.width=12, repr.plot.height=10)
DimPlot(adata_cluster_B, reduction='umap.wnn', group.by = 'condition', label=TRUE, label.size=6, repel=TRUE) +
ggtitle('WNN') + xlab('UMAP 1') + ylab('UMAP 2') + ggtitle('Combined')

In [ ]:
options(repr.plot.width=20, repr.plot.height=15)
#p1 <- DimPlot(adata, reduction='umap.wnn', group.by='donor_demux', split.by='donor_demux', label=FALSE, label.size=10, repel=TRUE)
#adata$value <- 1
p2 <- ggplot(adata_cluster_B[[]], aes(fill=condition, y=value, x=seurat_clusters)) + geom_bar(position=position_fill(reverse=TRUE), stat='identity') + xlab('') + ylab('percentage') + theme_light()
p2

In [ ]:
options(repr.plot.width=20, repr.plot.height=15)
#p1 <- DimPlot(adata, reduction='umap.wnn', group.by='donor_demux', split.by='donor_demux', label=FALSE, label.size=10, repel=TRUE)
#adata$value <- 1
p2 <- ggplot(adata_cluster_B[[]], aes(fill=donor_demux, y=value, x=seurat_clusters)) + geom_bar(position=position_fill(reverse=TRUE), stat='identity') + xlab('') + ylab('percentage') + theme_light()
p2

In [ ]:
marker.set <- cell.markers

#reMaking a dot plot with cell type markers. We will use this to not only assign cell types, but also narrow down the markers, since I have too many liver marker genes.
#Change width and height, since I'll have so many markers to visualize. (previous width/height was 25/10).
g = DotPlot(adata_cluster_B, assay='SCT', features=marker.set$marker, group.by='wnn_sub_cluster', col.min=0) +
        theme(axis.text.x=element_text(angle=45, hjust=1)) + xlab('') + ylab('')
    meta_summary = g$data
    colnames(meta_summary)[3] = "marker"
    meta_summary = merge(meta_summary, marker.set, by = "marker")

    options(repr.plot.width=20, repr.plot.height=10)
    figure <- ggplot(meta_summary, aes(x = marker, y = id)) +
      geom_point(aes(size = pct.exp, fill = avg.exp.scaled, stroke=NA),
                 shape = 21) +
      scale_size("% detected", range = c(0, 6)) +
      scale_fill_gradient(low = "lightgray", high = "blue",
                           guide = guide_colorbar(nbin = 200,
                                                  ticks.colour = "black", frame.colour = "black"),
                           name = "Average\nexpression") +
      ylab("Cluster") + xlab("") +
      theme_bw() +
      theme(axis.text = element_text(size = 100),
            axis.text.x = element_text(size = 20, angle = 45, hjust = 1, color = "black"),
            strip.text.x = element_text(size = 14),
            axis.text.y = element_text(size = 20, color = "black"),
            axis.title = element_text(size = 20)) +
      facet_nested(cols = vars(Compartment, CellType), scales = "free", space = "free")
figure

In [ ]:
options(repr.plot.width=10, repr.plot.height=20)
p1 <- VlnPlot(adata_cluster_B, features='nCount_SCT', group.by='wnn_sub_cluster', pt.size=0, log=TRUE) + geom_boxplot(width=.6, fill='white', alpha=.6) + geom_hline(yintercept=median(adata_cluster_B$nCount_SCT), linetype='dashed')
p2 <- VlnPlot(adata_cluster_B, features='nFeature_SCT', group.by='wnn_sub_cluster', pt.size=0, log=TRUE) + geom_boxplot(width=.6, fill='white', alpha=.6) + geom_hline(yintercept=median(adata_cluster_B$nFeature_SCT), linetype='dashed')
p3 <- VlnPlot(adata_cluster_B, features='nCount_ATAC', group.by='wnn_sub_cluster', pt.size=0, log=TRUE) + geom_boxplot(width=.6, fill='white', alpha=.6) + geom_hline(yintercept=median(adata_cluster_B$nCount_ATAC), linetype='dashed')
p4 <- VlnPlot(adata_cluster_B, features='nFeature_ATAC', group.by='wnn_sub_cluster', pt.size=0, log=TRUE) + geom_boxplot(width=.6, fill='white', alpha=.6) + geom_hline(yintercept=median(adata_cluster_B$nFeature_ATAC), linetype='dashed')
p1 / p2 / p3 / p4

In [ ]:
B.markers <- FindAllMarkers(adata_cluster_B, assay = 'SCT')

dim(B.markers)
head(B.markers)

In [ ]:
options(repr.plot.width=8, repr.plot.height=8)

B.plot.markers <- c('MS4A1','CD79A','BANK1','HLA-DQA1','MZB1','JCHAIN','IGHM','DERL3','TXNDC5')

for (mark in B.plot.markers) {
    p1 <- FeaturePlot(
      object = adata_cluster_B,
      reduction = "umap.wnn",
      features = c(mark),
      ncol = 1,
      raster=TRUE,
      order=T
    ) + ylim(c(2.5,10)) + xlim(c(-10,0)) + scale_colour_viridis_c(option='rocket', direction=-1)
    
    print(p1)
}

In [ ]:
save_data <- adata_cluster_B
save_data[["RNA"]] <- as(object = save_data[["RNA"]], Class = "Assay")
save_data[["SCT"]] <- as(object = save_data[["SCT"]], Class = "Assay")
save_data[["windows"]] <- as(object = save_data[["windows"]], Class = "Assay")
save_data[["RNA_raw"]] <- as(object = save_data[["RNA_raw"]], Class = "Assay")
saveRDS(save_data, '/nfs/lab/projects/nash_nafld_liver/downstream_all/windows_subclustering/240806_fnih_liver_ALL28lanes_ALL87donors_subclustering_B.Plasma_cells.rds')
saveRDS(adata_cluster_B, '/nfs/lab/projects/nash_nafld_liver/downstream_all/windows_subclustering/240806_fnih_liver_ALL28lanes_ALL87donors_subclustering_B.Plasma_cells_BPCells.rds')
save_data <- NULL
gc()

In [ ]:
write.table(B.markers, paste0('/nfs/lab/projects/nash_nafld_liver/downstream_all/windows_subclustering/240806_fnih_liver_ALL28lanes_ALL87donors_subclustering_B.Plasma_cells_cluster_markers.tsv'),
            sep='\t', quote=F, col.names=T, row.names=T)

In [ ]:
# Sinusoidal.Endothelial
barcodes.celltype.tmp = data.frame(barcode = WhichCells(adata_cluster_B, ident = c('3')),
                                   celltype = "B")
barcodes.celltype = rbind(barcodes.celltype, barcodes.celltype.tmp)

# Lymphatic.Endothelial
barcodes.celltype.tmp = data.frame(barcode = WhichCells(adata_cluster_B, ident = c('1','2')),
                                   celltype = "Plasma")
barcodes.celltype = rbind(barcodes.celltype, barcodes.celltype.tmp)

In [ ]:
dim(barcodes.celltype)
tail(barcodes.celltype)

# Cholangiocytes

In [ ]:
adata_cluster_cholangio <- subset(adata_sub, subset=seurat_clusters %in% as.character(c(14)))
adata_cluster_cholangio

In [ ]:
adata_cluster_cholangio <- FindClusters(adata_cluster_cholangio, graph.name='wsnn', algorithm=4, cluster.name='wnn_sub_cluster',
                              resolution = .25, verbose=TRUE, method = 'igraph')

In [ ]:
options(repr.plot.width=12, repr.plot.height=10)
DimPlot(adata_cluster_cholangio, reduction='umap.wnn', label=TRUE, label.size=6, repel=TRUE) +
ggtitle('WNN') + xlab('UMAP 1') + ylab('UMAP 2') + ggtitle('Combined')

In [ ]:
options(repr.plot.width=12, repr.plot.height=10)
DimPlot(adata_cluster_cholangio, reduction='umap.wnn', label=TRUE, label.size=6, repel=TRUE) +
ggtitle('WNN') + xlab('UMAP 1') + ylab('UMAP 2') + ggtitle('Combined')

In [ ]:
options(repr.plot.width=12, repr.plot.height=10)
DimPlot(adata_cluster_cholangio, reduction='umap.wnn', group.by = 'condition', label=TRUE, label.size=6, repel=TRUE) +
ggtitle('WNN') + xlab('UMAP 1') + ylab('UMAP 2') + ggtitle('Combined')

In [ ]:
options(repr.plot.width=20, repr.plot.height=15)
#p1 <- DimPlot(adata, reduction='umap.wnn', group.by='donor_demux', split.by='donor_demux', label=FALSE, label.size=10, repel=TRUE)
#adata$value <- 1
p2 <- ggplot(adata_cluster_cholangio[[]], aes(fill=condition, y=value, x=seurat_clusters)) + geom_bar(position=position_fill(reverse=TRUE), stat='identity') + xlab('') + ylab('percentage') + theme_light()
p2

In [ ]:
options(repr.plot.width=20, repr.plot.height=15)
#p1 <- DimPlot(adata, reduction='umap.wnn', group.by='donor_demux', split.by='donor_demux', label=FALSE, label.size=10, repel=TRUE)
#adata$value <- 1
p2 <- ggplot(adata_cluster_cholangio[[]], aes(fill=donor_demux, y=value, x=seurat_clusters)) + geom_bar(position=position_fill(reverse=TRUE), stat='identity') + xlab('') + ylab('percentage') + theme_light()
p2

In [ ]:
marker.set <- cell.markers

#reMaking a dot plot with cell type markers. We will use this to not only assign cell types, but also narrow down the markers, since I have too many liver marker genes.
#Change width and height, since I'll have so many markers to visualize. (previous width/height was 25/10).
g = DotPlot(adata_cluster_cholangio, assay='SCT', features=marker.set$marker, group.by='wnn_sub_cluster', col.min=0) +
        theme(axis.text.x=element_text(angle=45, hjust=1)) + xlab('') + ylab('')
    meta_summary = g$data
    colnames(meta_summary)[3] = "marker"
    meta_summary = merge(meta_summary, marker.set, by = "marker")

    options(repr.plot.width=20, repr.plot.height=10)
    figure <- ggplot(meta_summary, aes(x = marker, y = id)) +
      geom_point(aes(size = pct.exp, fill = avg.exp.scaled, stroke=NA),
                 shape = 21) +
      scale_size("% detected", range = c(0, 6)) +
      scale_fill_gradient(low = "lightgray", high = "blue",
                           guide = guide_colorbar(nbin = 200,
                                                  ticks.colour = "black", frame.colour = "black"),
                           name = "Average\nexpression") +
      ylab("Cluster") + xlab("") +
      theme_bw() +
      theme(axis.text = element_text(size = 100),
            axis.text.x = element_text(size = 20, angle = 45, hjust = 1, color = "black"),
            strip.text.x = element_text(size = 14),
            axis.text.y = element_text(size = 20, color = "black"),
            axis.title = element_text(size = 20)) +
      facet_nested(cols = vars(Compartment, CellType), scales = "free", space = "free")
figure

In [ ]:
options(repr.plot.width=10, repr.plot.height=20)
p1 <- VlnPlot(adata_cluster_cholangio, features='nCount_SCT', group.by='wnn_sub_cluster', pt.size=0, log=TRUE) + geom_boxplot(width=.6, fill='white', alpha=.6) + geom_hline(yintercept=median(adata_cluster_cholangio$nCount_SCT), linetype='dashed')
p2 <- VlnPlot(adata_cluster_cholangio, features='nFeature_SCT', group.by='wnn_sub_cluster', pt.size=0, log=TRUE) + geom_boxplot(width=.6, fill='white', alpha=.6) + geom_hline(yintercept=median(adata_cluster_cholangio$nFeature_SCT), linetype='dashed')
p3 <- VlnPlot(adata_cluster_cholangio, features='nCount_ATAC', group.by='wnn_sub_cluster', pt.size=0, log=TRUE) + geom_boxplot(width=.6, fill='white', alpha=.6) + geom_hline(yintercept=median(adata_cluster_cholangio$nCount_ATAC), linetype='dashed')
p4 <- VlnPlot(adata_cluster_cholangio, features='nFeature_ATAC', group.by='wnn_sub_cluster', pt.size=0, log=TRUE) + geom_boxplot(width=.6, fill='white', alpha=.6) + geom_hline(yintercept=median(adata_cluster_cholangio$nFeature_ATAC), linetype='dashed')
p1 / p2 / p3 / p4

In [ ]:
Cholangio.markers <- FindAllMarkers(adata_cluster_cholangio, assay = 'SCT')

dim(Cholangio.markers)
head(Cholangio.markers)

In [ ]:
options(repr.plot.width=8, repr.plot.height=8)

Cholangio.plot.markers <- c('MS4A1','CD79A','BANK1','HLA-DQA1','MZB1','JCHAIN','IGHM','DERL3','TXNDC5')

for (mark in Cholangio.plot.markers) {
    p1 <- FeaturePlot(
      object = adata_cluster_cholangio,
      reduction = "umap.wnn",
      features = c(mark),
      ncol = 1,
      raster=TRUE,
      order=T
    ) + scale_colour_viridis_c(option='rocket', direction=-1)
    
    print(p1)
}

In [ ]:
save_data <- adata_cluster_cholangio
save_data[["RNA"]] <- as(object = save_data[["RNA"]], Class = "Assay")
save_data[["SCT"]] <- as(object = save_data[["SCT"]], Class = "Assay")
save_data[["windows"]] <- as(object = save_data[["windows"]], Class = "Assay")
save_data[["RNA_raw"]] <- as(object = save_data[["RNA_raw"]], Class = "Assay")
saveRDS(save_data, '/nfs/lab/projects/nash_nafld_liver/downstream_all/windows_subclustering/240806_fnih_liver_ALL28lanes_ALL87donors_subclustering_Cholangio_cells.rds')
saveRDS(adata_cluster_cholangio, '/nfs/lab/projects/nash_nafld_liver/downstream_all/windows_subclustering/240806_fnih_liver_ALL28lanes_ALL87donors_subclustering_Cholangio_cells_BPCells.rds')
save_data <- NULL
gc()

In [ ]:
write.table(Cholangio.markers, paste0('/nfs/lab/projects/nash_nafld_liver/downstream_all/windows_subclustering/240806_fnih_liver_ALL28lanes_ALL87donors_subclustering_Cholangio_cells_cluster_markers.tsv'),
            sep='\t', quote=F, col.names=T, row.names=T)

In [ ]:
# Sinusoidal.Endothelial
barcodes.celltype.tmp = data.frame(barcode = WhichCells(adata_cluster_cholangio, ident = c('1','2','3','4')),
                                   celltype = "Cholangiocyte")
barcodes.celltype = rbind(barcodes.celltype, barcodes.celltype.tmp)

In [ ]:
dim(barcodes.celltype)
tail(barcodes.celltype)

# Hepatocytes

In [ ]:
adata_cluster_hepato <- subset(adata_sub, subset=seurat_clusters %in% as.character(c(1,4,5,6,7,8,9,10,17,19,21,22,25,26,29)))
adata_cluster_hepato

In [ ]:
adata_cluster_hepato <- FindClusters(adata_cluster_hepato, graph.name='wsnn', algorithm=4, cluster.name='wnn_sub_cluster',
                              resolution = .5, verbose=TRUE, method = 'igraph')

In [ ]:
options(repr.plot.width=12, repr.plot.height=10)
DimPlot(adata_cluster_hepato, reduction='umap.wnn', label=TRUE, label.size=6, repel=TRUE) +
ggtitle('WNN') + xlab('UMAP 1') + ylab('UMAP 2') + ggtitle('Combined')

In [ ]:
adata_cluster_hepato <- FindClusters(adata_cluster_hepato, graph.name='wsnn', algorithm=4, cluster.name='wnn_sub_cluster',
                              resolution = 1, verbose=TRUE, method = 'igraph')

In [ ]:
options(repr.plot.width=12, repr.plot.height=10)
DimPlot(adata_cluster_hepato, reduction='umap.wnn', label=TRUE, label.size=6, repel=TRUE) +
ggtitle('WNN') + xlab('UMAP 1') + ylab('UMAP 2') + ggtitle('Combined')

In [ ]:
options(repr.plot.width=24, repr.plot.height=4)
DimPlot(adata_cluster_hepato, reduction='umap.wnn', split.by='seurat_clusters', label=TRUE, label.size=6, repel=TRUE) +
ggtitle('WNN') + xlab('UMAP 1') + ylab('UMAP 2') + ggtitle('Combined')

In [ ]:
options(repr.plot.width=12, repr.plot.height=10)
DimPlot(adata_cluster_hepato, reduction='umap.wnn', group.by = 'condition', label=TRUE, label.size=6, repel=TRUE) +
ggtitle('WNN') + xlab('UMAP 1') + ylab('UMAP 2') + ggtitle('Combined')

In [ ]:
options(repr.plot.width=12, repr.plot.height=10)
DimPlot(adata_cluster_hepato, reduction='umap.wnn', group.by = 'donor_demux', label=TRUE, label.size=3, repel=TRUE) +
ggtitle('WNN') + xlab('UMAP 1') + ylab('UMAP 2') + ggtitle('Combined') + NoLegend()

In [ ]:
options(repr.plot.width=12, repr.plot.height=10)
DimPlot(adata_cluster_hepato, reduction='umap.wnn', group.by = 'Gender', label=TRUE, label.size=6, repel=TRUE) +
ggtitle('WNN') + xlab('UMAP 1') + ylab('UMAP 2') + ggtitle('Combined')

In [ ]:
options(repr.plot.width=20, repr.plot.height=15)
#p1 <- DimPlot(adata, reduction='umap.wnn', group.by='donor_demux', split.by='donor_demux', label=FALSE, label.size=10, repel=TRUE)
#adata$value <- 1
p2 <- ggplot(adata_cluster_hepato[[]], aes(fill=condition, y=value, x=seurat_clusters)) + geom_bar(position=position_fill(reverse=TRUE), stat='identity') + xlab('') + ylab('percentage') + theme_light(base_size=20)
p2

In [ ]:
options(repr.plot.width=25, repr.plot.height=15)
#p1 <- DimPlot(adata, reduction='umap.wnn', group.by='donor_demux', split.by='donor_demux', label=FALSE, label.size=10, repel=TRUE)
#adata$value <- 1
p2 <- ggplot(adata_cluster_hepato[[]], aes(fill=donor_demux, y=value, x=seurat_clusters)) + geom_bar(position=position_fill(reverse=TRUE), stat='identity') + xlab('') + ylab('percentage') + theme_light(base_size=20)
p2

In [ ]:
ptx.meta <- read.table('/nfs/lab/projects/nash_nafld_liver/donor_meta_20240812_fixed.tsv', sep='\t', header=T)
ptx.meta$Steatosis.grade <- factor(ptx.meta$Steatosis.grade)
ptx.meta

In [ ]:
which(ptx.meta$Fibrosis.stage.Brunt.Keliner.=='')

In [ ]:
ptx.meta$Fibrosis.stage.Brunt.Keliner.[76] <- NA

In [ ]:
adata_cluster_hepato[[]] <- left_join(adata_cluster_hepato[[]], ptx.meta, join_by(donor_demux==Patient.identifier))

In [ ]:
colnames(adata_cluster_hepato[[]])

In [ ]:
options(repr.plot.width=12, repr.plot.height=10)
DimPlot(adata_cluster_hepato, reduction='umap.wnn', group.by = 'Fibrosis.stage.Brunt.Keliner..y', label=TRUE, label.size=6, repel=TRUE) +
ggtitle('WNN') + xlab('UMAP 1') + ylab('UMAP 2') + ggtitle('Combined')

In [ ]:
options(repr.plot.width=20, repr.plot.height=15)
#p1 <- DimPlot(adata, reduction='umap.wnn', group.by='donor_demux', split.by='donor_demux', label=FALSE, label.size=10, repel=TRUE)
#adata$value <- 1
p2 <- ggplot(adata_cluster_hepato[[]], aes(fill=Gender, y=value, x=seurat_clusters)) + geom_bar(position=position_fill(reverse=TRUE), stat='identity') + xlab('') + ylab('percentage') + theme_light(base_size=20)
p2

In [ ]:
adata_cluster_hepato[[]]

In [ ]:
marker.set <- cell.markers

#reMaking a dot plot with cell type markers. We will use this to not only assign cell types, but also narrow down the markers, since I have too many liver marker genes.
#Change width and height, since I'll have so many markers to visualize. (previous width/height was 25/10).
g = DotPlot(adata_cluster_hepato, assay='SCT', features=marker.set$marker, group.by='wnn_sub_cluster', col.min=0) +
        theme(axis.text.x=element_text(angle=45, hjust=1)) + xlab('') + ylab('')
    meta_summary = g$data
    colnames(meta_summary)[3] = "marker"
    meta_summary = merge(meta_summary, marker.set, by = "marker")

    options(repr.plot.width=20, repr.plot.height=10)
    figure <- ggplot(meta_summary, aes(x = marker, y = id)) +
      geom_point(aes(size = pct.exp, fill = avg.exp.scaled, stroke=NA),
                 shape = 21) +
      scale_size("% detected", range = c(0, 6)) +
      scale_fill_gradient(low = "lightgray", high = "blue",
                           guide = guide_colorbar(nbin = 200,
                                                  ticks.colour = "black", frame.colour = "black"),
                           name = "Average\nexpression") +
      ylab("Cluster") + xlab("") +
      theme_bw() +
      theme(axis.text = element_text(size = 100),
            axis.text.x = element_text(size = 20, angle = 45, hjust = 1, color = "black"),
            strip.text.x = element_text(size = 14),
            axis.text.y = element_text(size = 20, color = "black"),
            axis.title = element_text(size = 20)) +
      facet_nested(cols = vars(Compartment, CellType), scales = "free", space = "free")
figure

In [ ]:
options(repr.plot.width=10, repr.plot.height=20)
p1 <- VlnPlot(adata_cluster_hepato, features='nCount_SCT', group.by='wnn_sub_cluster', pt.size=0, log=TRUE) + geom_boxplot(width=.6, fill='white', alpha=.6) + geom_hline(yintercept=median(adata_cluster_hepato$nCount_SCT), linetype='dashed')
p2 <- VlnPlot(adata_cluster_hepato, features='nFeature_SCT', group.by='wnn_sub_cluster', pt.size=0, log=TRUE) + geom_boxplot(width=.6, fill='white', alpha=.6) + geom_hline(yintercept=median(adata_cluster_hepato$nFeature_SCT), linetype='dashed')
p3 <- VlnPlot(adata_cluster_hepato, features='nCount_ATAC', group.by='wnn_sub_cluster', pt.size=0, log=TRUE) + geom_boxplot(width=.6, fill='white', alpha=.6) + geom_hline(yintercept=median(adata_cluster_hepato$nCount_ATAC), linetype='dashed')
p4 <- VlnPlot(adata_cluster_hepato, features='nFeature_ATAC', group.by='wnn_sub_cluster', pt.size=0, log=TRUE) + geom_boxplot(width=.6, fill='white', alpha=.6) + geom_hline(yintercept=median(adata_cluster_hepato$nFeature_ATAC), linetype='dashed')
p1 / p2 / p3 / p4

In [ ]:
hepato.markers <- FindAllMarkers(adata_cluster_hepato, assay = 'SCT')

dim(hepato.markers)
head(hepato.markers)

In [ ]:
options(repr.plot.width=8, repr.plot.height=8)

hepato.plot.markers <- c('HAL','ARG1','CPS1','HAMP','APOC1','ADH4','CYP2E1','AHR','CYP3A4')

for (mark in hepato.plot.markers) {
    p1 <- FeaturePlot(
      object = adata_cluster_hepato,
      reduction = "umap.wnn",
      features = c(mark),
      ncol = 1,
      raster=TRUE,
      order=T
    ) + scale_colour_viridis_c(option='rocket', direction=-1)
    
    print(p1)
}

In [ ]:
options(repr.plot.width=8, repr.plot.height=8)

hepato.plot.markers <- c('HAL','ARG1','CPS1','HAMP','APOC1','ADH4','CYP2E1','AHR','CYP3A4')

for (mark in hepato.plot.markers) {
    p1 <- VlnPlot(sort = T,
      object = adata_cluster_hepato,
      features = c(mark))
    
    print(p1)
}

In [ ]:
options(repr.plot.width=8, repr.plot.height=8)

hepato.plot.markers <- c('CDKN2A', 'CDKN1A', 'BCL2L1', 'CTSB', 'CCL20')

for (mark in hepato.plot.markers) {
    p1 <- FeaturePlot(
      object = adata_cluster_hepato,
      reduction = "umap.wnn",
      features = c(mark),
      ncol = 1,
      raster=TRUE,
      order=T
    ) + scale_colour_viridis_c(option='rocket', direction=-1)
    
    print(p1)
}

In [ ]:
options(repr.plot.width=8, repr.plot.height=8)

hepato.plot.markers <- c('MKI67')

for (mark in hepato.plot.markers) {
    p1 <- FeaturePlot(
      object = adata_cluster_hepato,
      reduction = "umap.wnn",
      features = c(mark),
      ncol = 1,
      raster=TRUE,
      order=T
    ) + scale_colour_viridis_c(option='rocket', direction=-1)
    
    print(p1)
}

In [ ]:
options(repr.plot.width=8, repr.plot.height=8)

hepato.plot.markers <- c('XIST', 'PZP','ESR1')

for (mark in hepato.plot.markers) {
    p1 <- FeaturePlot(
      object = adata_cluster_hepato,
      reduction = "umap.wnn",
      features = c(mark),
      ncol = 1,
      raster=TRUE,
      order=T
    ) + scale_colour_viridis_c(option='rocket', direction=-1)
    
    print(p1)
}

In [ ]:
options(repr.plot.width=8, repr.plot.height=8)

hepato.plot.markers <- c('HSPB1')

for (mark in hepato.plot.markers) {
    p1 <- FeaturePlot(
      object = adata_cluster_hepato,
      reduction = "umap.wnn",
      features = c(mark),
      ncol = 1,
      raster=TRUE,
      order=T
    ) + scale_colour_viridis_c(option='rocket', direction=-1)
    
    print(p1)
}

In [ ]:
options(repr.plot.width=8, repr.plot.height=8)

hepato.plot.markers <- c('SERPINE1')

for (mark in hepato.plot.markers) {
    p1 <- FeaturePlot(
      object = adata_cluster_hepato,
      reduction = "umap.wnn",
      features = c(mark),
      ncol = 1,
      raster=TRUE,
      order=T
    ) + scale_colour_viridis_c(option='rocket', direction=-1)
    
    print(p1)
}

In [ ]:
options(repr.plot.width=8, repr.plot.height=8)

hepato.plot.markers <- c('HGF')

for (mark in hepato.plot.markers) {
    p1 <- FeaturePlot(
      object = adata_cluster_hepato,
      reduction = "umap.wnn",
      features = c(mark),
      ncol = 1,
      raster=TRUE,
      order=T
    ) + scale_colour_viridis_c(option='rocket', direction=-1)
    
    print(p1)
}

In [ ]:
options(repr.plot.width=8, repr.plot.height=8)

hepato.plot.markers <- c('ICAM1','KRT7','AFP','EPCAM','NCAM1')

for (mark in hepato.plot.markers) {
    p1 <- FeaturePlot(
      object = adata_cluster_hepato,
      reduction = "umap.wnn",
      features = c(mark),
      ncol = 1,
      raster=TRUE,
      order=T
    ) + scale_colour_viridis_c(option='rocket', direction=-1)
    
    print(p1)
}

In [ ]:
options(repr.plot.width=8, repr.plot.height=8)

hepato.plot.markers <- c('HIF1A')

for (mark in hepato.plot.markers) {
    p1 <- FeaturePlot(
      object = adata_cluster_hepato,
      reduction = "umap.wnn",
      features = c(mark),
      ncol = 1,
      raster=TRUE,
      order=T
    ) + scale_colour_viridis_c(option='rocket', direction=-1)
    
    print(p1)
}

In [ ]:
Zone.1 - 1,3,6,14,18 (9,8?)
Zone.2 - 10
Zone.3 - 2,7,17
Senescent - 5
Cholesterol/Female - 4
8
9 - Fatty Acid Metabolism
11 - Drug detox? P450
12 - Heat Shock
13 - 
15 - Donor specific
16 - Metabolic

In [ ]:
4 - Cholesterol? Female enriched, XIST, Estrogen receptor
8 - Female enriched
9 - Fatty Acid Metabolism
11 - Drug detox? P450
12 - Heat Shock
13 - Partially Specific - HL160017 - Maybe zone 1?
15 - Specific - HL180069 - Maybe zone 1?
16 - Metabolic (maybe zone 1)
17 - DNA synthesis
18 - ??

9,11-16 are disease enriched specific

In [ ]:
table(Idents(adata_cluster_hepato))

In [ ]:
table(adata_cluster_hepato[[]][c('donor_demux','seurat_clusters')])

In [ ]:
table(adata_cluster_hepato[[]][c('seurat_clusters','condition')])

In [ ]:
HSP clusters – Maybe 9 or 10
Hypoxia
Zone 3 – Wnt/Catenin, Glutamine Synthase
Zone 1 – YAP, HN4A, gluconeogenesis, fatty acid degradation, urea

8 – enriched in NAFL

Senescent – 4.5% in NASH liver

In [ ]:
head(filter(hepato.markers, cluster=='4' & pct.1 > pct.2), n=20)
head(filter(hepato.markers, cluster=='4' & pct.1 > pct.2), n=20)$gene

In [ ]:
options(repr.plot.width=16, repr.plot.height=6)

DEenrichRPlot(
  adata_cluster_hepato,
  ident.1 = '4',
  ident.2 = NULL,
  balanced = TRUE,
  logfc.threshold = 0.25,
  assay = 'SCT',
  max.genes=1000,
  test.use = "wilcox",
  p.val.cutoff = 0.05,
  cols = NULL,
  enrich.database = 'GO_Biological_Process_2023',
  num.pathway = 10,
  return.gene.list = FALSE
)

In [ ]:
head(filter(hepato.markers, cluster=='8' & pct.1 > pct.2), n=20)
head(filter(hepato.markers, cluster=='8' & pct.1 > pct.2), n=20)$gene

In [ ]:
options(repr.plot.width=16, repr.plot.height=6)

DEenrichRPlot(
  adata_cluster_hepato,
  ident.1 = '8',
  ident.2 = NULL,
  balanced = TRUE,
  logfc.threshold = 0.25,
  assay = 'SCT',
  max.genes=1000,
  test.use = "wilcox",
  p.val.cutoff = 0.05,
  cols = NULL,
  enrich.database = 'GO_Biological_Process_2023',
  num.pathway = 10,
  return.gene.list = FALSE
)

In [ ]:
head(filter(hepato.markers, cluster=='8' & pct.1 > pct.2), n=30)
head(filter(hepato.markers, cluster=='8' & pct.1 > pct.2), n=30)$gene

In [ ]:
head(filter(hepato.markers, cluster=='9' & pct.1 > pct.2), n=20)
head(filter(hepato.markers, cluster=='9' & pct.1 > pct.2), n=20)$gene

In [ ]:
options(repr.plot.width=16, repr.plot.height=6)
DEenrichRPlot(
  adata_cluster_hepato,
  ident.1 = '9',
  ident.2 = NULL,
  balanced = TRUE,
  logfc.threshold = 0.25,
  assay = 'SCT',
  max.genes=1000,
  test.use = "wilcox",
  p.val.cutoff = 0.05,
  cols = NULL,
  enrich.database = 'GO_Biological_Process_2023',
  num.pathway = 10,
  return.gene.list = FALSE
)

In [ ]:
head(filter(hepato.markers, cluster=='11' & pct.1 > pct.2), n=20)
head(filter(hepato.markers, cluster=='11' & pct.1 > pct.2), n=20)$gene

In [ ]:
options(repr.plot.width=16, repr.plot.height=6)
DEenrichRPlot(
  adata_cluster_hepato,
  ident.1 = '11',
  ident.2 = NULL,
  balanced = TRUE,
  logfc.threshold = 0.25,
  assay = 'SCT',
  max.genes=1000,
  test.use = "wilcox",
  p.val.cutoff = 0.05,
  cols = NULL,
  enrich.database = 'GO_Biological_Process_2023',
  num.pathway = 10,
  return.gene.list = FALSE
)

In [ ]:
head(filter(hepato.markers, cluster=='12' & pct.1 > pct.2), n=20)
head(filter(hepato.markers, cluster=='12' & pct.1 > pct.2), n=20)$gene

In [ ]:
options(repr.plot.width=16, repr.plot.height=6)
DEenrichRPlot(
  adata_cluster_hepato,
  ident.1 = '12',
  ident.2 = NULL,
  balanced = TRUE,
  logfc.threshold = 0.25,
  assay = 'SCT',
  max.genes=1000,
  test.use = "wilcox",
  p.val.cutoff = 0.05,
  cols = NULL,
  enrich.database = 'GO_Biological_Process_2023',
  num.pathway = 10,
  return.gene.list = FALSE
)

In [ ]:
head(filter(hepato.markers, cluster=='13' & pct.1 > pct.2), n=20)
head(filter(hepato.markers, cluster=='13' & pct.1 > pct.2), n=20)$gene

In [ ]:
options(repr.plot.width=16, repr.plot.height=6)

DEenrichRPlot(
  adata_cluster_hepato,
  ident.1 = '13',
  ident.2 = NULL,
  balanced = TRUE,
  logfc.threshold = 0.25,
  assay = 'SCT',
  max.genes=1000,
  test.use = "wilcox",
  p.val.cutoff = 0.05,
  cols = NULL,
  enrich.database = 'GO_Biological_Process_2023',
  num.pathway = 10,
  return.gene.list = FALSE
)

In [ ]:
head(filter(hepato.markers, cluster=='14' & pct.1 > pct.2), n=20)
head(filter(hepato.markers, cluster=='14' & pct.1 > pct.2), n=20)$gene

In [ ]:
options(repr.plot.width=16, repr.plot.height=6)

DEenrichRPlot(
  adata_cluster_hepato,
  ident.1 = '14',
  ident.2 = NULL,
  balanced = TRUE,
  logfc.threshold = 0.25,
  assay = 'SCT',
  max.genes=1000,
  test.use = "wilcox",
  p.val.cutoff = 0.05,
  cols = NULL,
  enrich.database = 'GO_Biological_Process_2023',
  num.pathway = 10,
  return.gene.list = FALSE
)

In [ ]:
head(filter(hepato.markers, cluster=='15' & pct.1 > pct.2), n=20)
head(filter(hepato.markers, cluster=='15' & pct.1 > pct.2), n=20)$gene

In [ ]:
options(repr.plot.width=16, repr.plot.height=6)

DEenrichRPlot(
  adata_cluster_hepato,
  ident.1 = '15',
  ident.2 = NULL,
  balanced = TRUE,
  logfc.threshold = 0.25,
  assay = 'SCT',
  max.genes=1000,
  test.use = "wilcox",
  p.val.cutoff = 0.05,
  cols = NULL,
  enrich.database = 'GO_Biological_Process_2023',
  num.pathway = 10,
  return.gene.list = FALSE
)

In [ ]:
head(filter(hepato.markers, cluster=='16' & pct.1 > pct.2), n=20)
head(filter(hepato.markers, cluster=='16' & pct.1 > pct.2), n=20)$gene

In [ ]:
options(repr.plot.width=16, repr.plot.height=6)

DEenrichRPlot(
  adata_cluster_hepato,
  ident.1 = '16',
  ident.2 = NULL,
  balanced = TRUE,
  logfc.threshold = 0.25,
  assay = 'SCT',
  max.genes=1000,
  test.use = "wilcox",
  p.val.cutoff = 0.05,
  cols = NULL,
  enrich.database = 'GO_Biological_Process_2023',
  num.pathway = 10,
  return.gene.list = FALSE
)

In [ ]:
head(filter(hepato.markers, cluster=='17' & pct.1 > pct.2), n=20)
head(filter(hepato.markers, cluster=='17' & pct.1 > pct.2), n=20)$gene

In [ ]:
head(filter(hepato.markers, cluster=='18' & pct.1 > pct.2), n=20)
head(filter(hepato.markers, cluster=='18' & pct.1 > pct.2), n=20)$gene

In [ ]:
# Hepatocytes
barcodes.celltype.tmp = data.frame(barcode = Cells(adata_cluster_hepato),
                                   celltype = "Hepatocytes")
barcodes.celltype = rbind(barcodes.celltype, barcodes.celltype.tmp)

In [ ]:
ALB.mat <- as.matrix(adata_cluster_hepato$SCT$counts['ALB',])
ALB.mat[,which(ALB.mat > 250)]
adata_cluster_hepato[[]][names(ALB.mat[,which(ALB.mat > 250)]),]

In [ ]:
barcodes.celltype <- read.table('/nfs/lab/projects/nash_nafld_liver/downstream_all/08052024_fnih_liver_ALL28lanes_ALL87donors_50kHVWsby16pool_HarmonyCovariates_DonorDemuxAndPoolingBatch_filtered_amulet_res10_5pctamulet_subset_clustered_>10bc_incomplete_recluster_celltypes_sub.tsv',
            header=T, sep='\t')

dim(barcodes.celltype)
head(barcodes.celltype)

In [ ]:
length(rownames(adata_cluster_hepato[[]][adata_cluster_hepato[[]]$seurat_clusters %in% c('1','3','4','6','8','9','11','14','18'),]))
head(rownames(adata_cluster_hepato[[]][adata_cluster_hepato[[]]$seurat_clusters %in% c('1','3','6','11','14','18'),]))

In [ ]:
rownames(barcodes.celltype) <- barcodes.celltype$barcode

barcodes.celltype[rownames(adata_cluster_hepato[[]][adata_cluster_hepato[[]]$seurat_clusters %in% c('1','3','4','6','8','9','11','14','16','18'),]),]$subtype <- 'Zone.1.Hepatocytes'
barcodes.celltype[rownames(adata_cluster_hepato[[]][adata_cluster_hepato[[]]$seurat_clusters %in% c('10'),]),]$subtype <- 'Zone.2.Hepatocytes'
barcodes.celltype[rownames(adata_cluster_hepato[[]][adata_cluster_hepato[[]]$seurat_clusters %in% c('2','7','17'),]),]$subtype <- 'Zone.3.Hepatocytes'
barcodes.celltype[rownames(adata_cluster_hepato[[]][adata_cluster_hepato[[]]$seurat_clusters %in% c('5'),]),]$subtype <- 'Senescent.Hepatocytes'
barcodes.celltype[rownames(adata_cluster_hepato[[]][adata_cluster_hepato[[]]$seurat_clusters %in% c('12'),]),]$subtype <- 'Heat.Shock.Hepatocytes'
barcodes.celltype[rownames(adata_cluster_hepato[[]][adata_cluster_hepato[[]]$seurat_clusters %in% c('13'),]),]$subtype <- 'Amino.Acid.Synthesis.Hepatocytes'
barcodes.celltype[rownames(adata_cluster_hepato[[]][adata_cluster_hepato[[]]$seurat_clusters %in% c('15'),]),]$subtype <- 'Apoptotic.Hepatocytes'

In [ ]:
write.table(barcodes.celltype, '/nfs/lab/projects/nash_nafld_liver/downstream_all/08052024_fnih_liver_ALL28lanes_ALL87donors_50kHVWsby16pool_HarmonyCovariates_DonorDemuxAndPoolingBatch_filtered_amulet_res10_5pctamulet_subset_clustered_>10bc_incomplete_recluster_celltypes_sub_hepato.tsv',
            row.names=F, col.names=T, quote=F, sep='\t')

# Small cell types

In [ ]:
# Mast
barcodes.celltype.tmp = data.frame(barcode = WhichCells(adata_sub, ident = c('28')),
                                   celltype = "Mast")
barcodes.celltype = rbind(barcodes.celltype, barcodes.celltype.tmp)

# Erythroblast
barcodes.celltype.tmp = data.frame(barcode = WhichCells(adata_sub, ident = c('30')),
                                   celltype = "Erythroblast")
barcodes.celltype = rbind(barcodes.celltype, barcodes.celltype.tmp)

# Schwann
barcodes.celltype.tmp = data.frame(barcode = WhichCells(adata_sub, ident = c('27')),
                                   celltype = "Schwann")
barcodes.celltype = rbind(barcodes.celltype, barcodes.celltype.tmp)

In [ ]:
dim(barcodes.celltype)

In [ ]:
adata_sub

In [ ]:
adata_sub

# Save out intermediate for colab

In [ ]:
adata_sub

In [ ]:
adata_sub$celltype <- NULL
adata_sub$subtype <- NULL

adata_sub[[]] <- cbind(adata_sub[[]], barcodes.celltype[rownames(adata_sub[[]]),])

In [ ]:
#CHANGE NOTHING
#Making UMAPs that are colored by cluster, will need to assign cell types later. 
options(repr.plot.width=18, repr.plot.height=6)
p1 <- DimPlot(adata_sub, reduction='umap.rna', group.by='celltype', label=TRUE, label.size=6, repel=TRUE) + ggtitle('RNA')
p1 <- p1 + xlab('UMAP 1') + ylab('UMAP 2') + ggtitle('RNA only')
p2 <- DimPlot(adata_sub, reduction='umap.atac', group.by='celltype', label=TRUE, label.size=6, repel=TRUE) + ggtitle('ATAC')
p2 <- p2 + xlab('UMAP 1') + ylab('UMAP 2') + ggtitle('ATAC only')
p3 <- DimPlot(adata_sub, reduction='umap.wnn', group.by='celltype', label=TRUE, label.size=6, repel=TRUE) + ggtitle('WNN')
p3 <- p3 + xlab('UMAP 1') + ylab('UMAP 2') + ggtitle('Combined')
p1 + p2 + p3 & NoLegend() & theme(plot.title=element_text(hjust=0.5))

In [ ]:
#CHANGE NOTHING
#Making UMAPs that are colored by cluster, will need to assign cell types later. 
options(repr.plot.width=18, repr.plot.height=6)
p1 <- DimPlot(adata_sub, reduction='umap.rna', group.by='subtype', label=TRUE, label.size=4, repel=TRUE) + ggtitle('RNA')
p1 <- p1 + xlab('UMAP 1') + ylab('UMAP 2') + ggtitle('RNA only')
p2 <- DimPlot(adata_sub, reduction='umap.atac', group.by='subtype', label=TRUE, label.size=4, repel=TRUE) + ggtitle('ATAC')
p2 <- p2 + xlab('UMAP 1') + ylab('UMAP 2') + ggtitle('ATAC only')
p3 <- DimPlot(adata_sub, reduction='umap.wnn', group.by='subtype', label=TRUE, label.size=4, repel=TRUE) + ggtitle('WNN')
p3 <- p3 + xlab('UMAP 1') + ylab('UMAP 2') + ggtitle('Combined')
p1 + p2 + p3 & NoLegend() & theme(plot.title=element_text(hjust=0.5))

In [ ]:
barcodes.celltype

In [ ]:
missing.barcodes = setdiff(names(Idents(adata_sub)), barcodes.celltype$barcode)
length(missing.barcodes)

In [ ]:
# Add a column metadata
adata_sub@meta.data$celltypes.detailed <- barcodes.celltype$celltype[match(rownames(adata_sub@meta.data), barcodes.celltype$barcode)]

In [ ]:
options(repr.plot.width=12, repr.plot.height=10)
DimPlot(adata_sub, reduction='umap.wnn', group.by = 'celltypes.detailed', label=TRUE, label.size=6, repel=TRUE) +
ggtitle('WNN') + xlab('UMAP 1') + ylab('UMAP 2') + ggtitle('Combined')

In [ ]:
write.table(barcodes.celltype, '/nfs/lab/projects/nash_nafld_liver/downstream_all/08052024_fnih_liver_ALL28lanes_ALL87donors_50kHVWsby16pool_HarmonyCovariates_DonorDemuxAndPoolingBatch_filtered_amulet_res10_5pctamulet_subset_clustered_>10bc_incomplete_recluster_celltypes.tsv',
            row.names=F, col.names=T, quote=F, sep='\t')

In [ ]:
saveRDS(adata_sub, '/nfs/lab/projects/nash_nafld_liver/downstream_all/08052024_fnih_liver_ALL28lanes_ALL87donors_50kHVWsby16pool_HarmonyCovariates_DonorDemuxAndPoolingBatch_filtered_amulet_res10_5pctamulet_subset_clustered_>10bc_incomplete_recluster.rds')

In [ ]:
barcodes.celltype$subtype <- barcodes.celltype$celltype

In [ ]:
unique(barcodes.celltype$celltype)

In [ ]:
barcodes.celltype[barcodes.celltype$celltype=='Sinusoidal.Endothelial',]$celltype <- 'Endothelial'
barcodes.celltype[barcodes.celltype$celltype=='Lymphatic.Endothelial',]$celltype <- 'Endothelial'
barcodes.celltype[barcodes.celltype$celltype=='Central.Vein.Endothelial',]$celltype <- 'Endothelial'
barcodes.celltype[barcodes.celltype$celltype=='Vascular.Endothelial',]$celltype <- 'Endothelial'

barcodes.celltype[barcodes.celltype$celltype=='Quescent.HSC',]$celltype <- 'HSC'
barcodes.celltype[barcodes.celltype$celltype=='Activated.HSC',]$celltype <- 'HSC'
barcodes.celltype[barcodes.celltype$celltype=='Mesenchymal',]$celltype <- 'HSC'

barcodes.celltype[barcodes.celltype$celltype=='Kupffer',]$celltype <- 'Myeloid'
barcodes.celltype[barcodes.celltype$celltype=='Macrophage',]$celltype <- 'Myeloid'
barcodes.celltype[barcodes.celltype$celltype=='Lipid.Associated.Macrophage',]$celltype <- 'Myeloid'

barcodes.celltype[barcodes.celltype$celltype=='Plasma',]$celltype <- 'B'

In [ ]:
write.table(barcodes.celltype, '/nfs/lab/projects/nash_nafld_liver/downstream_all/08052024_fnih_liver_ALL28lanes_ALL87donors_50kHVWsby16pool_HarmonyCovariates_DonorDemuxAndPoolingBatch_filtered_amulet_res10_5pctamulet_subset_clustered_>10bc_incomplete_recluster_celltypes_sub.tsv',
            row.names=F, col.names=T, quote=F, sep='\t')

In [ ]:
head(barcodes.celltype)

# Call Peaks  
This needs to be run in a different environment, and I've had the most success running from a terminal. Command placed here but run separately.

In [ ]:
# Start a screen
mamba activate macs2_signac
R

In [ ]:
library(Signac)
barcodes.celltype <- read.table('/nfs/lab/projects/nash_nafld_liver/downstream_all/08052024_fnih_liver_ALL28lanes_ALL87donors_50kHVWsby16pool_HarmonyCovariates_DonorDemuxAndPoolingBatch_filtered_amulet_res10_5pctamulet_subset_clustered_>10bc_incomplete_recluster_celltypes.tsv',
            header=T, sep='\t')

In [ ]:
adata <- readRDS('/nfs/lab/projects/nash_nafld_liver/keep_demux/06052024_fnih_liver_ALL28lanes_ALL87donors_50kHVWsby16pool_HarmonyCovariates_DonorDemuxAndPoolingBatch_filtered_amulet.rds')
adata

In [ ]:
adata$RNA <- NULL
adata$SCT <- NULL
adata$RNA_raw <- NULL
gc()

In [ ]:
adata <- subset(adata, cells=barcodes.celltype$barcode)
adata
gc()

In [ ]:
adata@meta.data$celltypes.detailed <- barcodes.celltype$celltype[match(rownames(adata@meta.data), barcodes.celltype$barcode)]

In [ ]:
peaks <- CallPeaks(
    object = adata,
    group.by = "celltypes.detailed",
    assay='windows',
    outdir='/nfs/lab/projects/nash_nafld_liver/downstream_all/windows_peak_call/',
    macs2.path='/home/welison/.conda/envs/mamba/envs/macs2_signac/bin/macs2',
    cleanup=FALSE,
    verbose=TRUE
)

In [ ]:
saveRDS(peaks, '/nfs/lab/projects/nash_nafld_liver/downstream_all/windows_peak_call/Signac_GRanges.RDS')

#### Union peaks

#Bash

#First in bash to remove GIANT header
mapfile -t cells < listofcells.txt
for cell in ${cells[@]}; do tail -n +26 ${cell}_peaks.xls > ${cell}.noheader.bed; done

In [ ]:
#Now in R

##DO THIS OR IT WILL RANDOMLY PUT YOUR COORDINATES IN SCIENTIFIC NOTATION\n"
options(scipen=999)

In [ ]:
#Read in peak lists
cells<-scan('/nfs/lab/projects/nash_nafld_liver/downstream_all/windows_peak_call/union_peaks/listofcells.txt', sep="\n", what="")
orgpeaks<-list()
for (cell in cells){
    orgpeaks[[cell]]<-read.table(sprintf('/nfs/lab/projects/nash_nafld_liver/downstream_all/windows_peak_call/union_peaks/%s.noheader.bed',cell), sep="\t", header=TRUE)
}
head(orgpeaks[[1]])

In [ ]:
#Shrink peaks greater than 300bp down to 300bp
shrunkpeaks<-lapply(1:length(cells), function(i){
    df<-orgpeaks[[i]]
    for (j in 1:nrow(df)){
        psize<-df$length[j]
        pstart<-df$start[j]
        pend<-df$end[j]
        if (psize > 300){
            summit<-df$abs_summit[j]
            newstart<-as.numeric(summit)-150
            newend<-as.numeric(summit)+150
            df$start[j]<-newstart
            df$end[j]<-newend
            df$length[j]<-newend-newstart
        }

    }
    return(df)
})

In [ ]:
#Make sure cell names match then add names back into list
##NOTE: Some of the string parsing here will dependon your naming convention! In this case, my string is Celltype_peak_#
for (i in 1:length(cells)){
    df<-shrunkpeaks[[i]]
    string<-df[1,10]
    ct<-sub("_.*", "", string)
    ctorder<-cells[i]
    message(paste0("Does ", ct, " match ", ctorder,"?"))
} 

In [ ]:
#Add names back in (these get lost during lapply)
names(shrunkpeaks)<-cells

In [ ]:
#Save peaks
for (i in 1:length(cells)){
    df<-shrunkpeaks[[i]]
    ct<-names(shrunkpeaks)[i]
    write.table(df, sprintf('/nfs/lab/projects/nash_nafld_liver/downstream_all/windows_peak_call/union_peaks/%s.shrunkpeaks.bed',ct), row.names=FALSE, sep="\t", quote=FALSE)
}

In [ ]:
#Basically everything else is now bash
#Modify input to be ready for bedops

for cell in ${cells[@]}; do awk '{print $1,$2,$3,$10,$6}' OFS="\t" ${cell}.shrunkpeaks.bed | tail -n +2 > ${cell}.mod.shrunkpeaks.bed; done

#!/bin/bash
##Define directory locations
#input directory must contain files with .bed extension to be analyzed
indir=/nfs/lab/projects/nash_nafld_liver/downstream_all/windows_peak_call/union_peaks/
tmpdir=/nfs/lab/projects/nash_nafld_liver/downstream_all/windows_peak_call/union_peaks/tmpdir/
outdir=/nfs/lab/projects/nash_nafld_liver/downstream_all/windows_peak_call/union_peaks/
#Name your output
outfile=LiverUnionPeaks.bed
#Initialize iteration, output and stop clause
touch ${outdir}/${outfile}
iters=1
stop=0

#Make tmp directory
if [ ! -d $tmpdir ]; then
mkdir $tmpdir;
fi

#Merge all bedfiles into one file
#NEED TO CHANGE THIS IF YOU HAVE MULTIPLE BED FILES IN HERE YOU DON'T WANT TO ANALYZE
bedops -u ${indir}/*.bed > $tmpdir/tmp.bed

while [ $stop == 0 ]
do 
echo "merge steps..."
#Merge overlapping peaks in this input file
bedops -m $tmpdir/tmp.bed > ${tmpdir}/tmpmerge.bed

#Find the peak in each merged cluster with the highest read
bedmap --max-element $tmpdir/tmpmerge.bed $tmpdir/tmp.bed \
| sort-bed - \
> $tmpdir/${iters}.bed \

#Add recent iteration to final bed
#Used to write each of these outputs out itndividually and then merge but it takes up WAY too much space
cat ${outdir}/${outfile} $tmpdir/${iters}.bed > ${outdir}/tmp && mv ${outdir}/tmp ${outdir}/${outfile}

num=$(wc -l $tmpdir/${iters}.bed | awk '{print $1}')
echo "Adding ${num} elements"
#Find which peaks don't overlap the previously defined "true" peak for each merged cluster
bedops -n 1 $tmpdir/tmp.bed $tmpdir/${iters}.bed \
> $tmpdir/tmp2.bed
#Make these peaks your new starting file
mv $tmpdir/tmp2.bed $tmpdir/tmp.bed
#Clean up tmp file (if I let it go this tmp directory gets MASSIVE AF)
rm $tmpdir/${iters}.bed

#Checks if the file is empty
if [ ${num} == 0 ]
then
stop=1
fi

((iters++))
done

#Add headers back in
echo -e "Chr\tStart\tEnd\tPeakID\tPileup_Score" > header && cat header ${outdir}/${outfile} > tmp && mv tmp ${outdir}/${outfile}

rm -r $tmpdir

In [ ]:
library(Signac)
library(GenomicRanges)
library(GenomeInfoDb)

barcodes.celltype <- read.table('/nfs/lab/projects/nash_nafld_liver/downstream_all/08052024_fnih_liver_ALL28lanes_ALL87donors_50kHVWsby16pool_HarmonyCovariates_DonorDemuxAndPoolingBatch_filtered_amulet_res10_5pctamulet_subset_clustered_>10bc_incomplete_recluster_celltypes.tsv',
            header=T, sep='\t')

adata <- readRDS('/nfs/lab/projects/nash_nafld_liver/keep_demux/06052024_fnih_liver_ALL28lanes_ALL87donors_50kHVWsby16pool_HarmonyCovariates_DonorDemuxAndPoolingBatch_filtered_amulet.rds')
adata

adata$RNA <- NULL
adata$SCT <- NULL
adata$RNA_raw <- NULL
gc()

adata <- subset(adata, cells=barcodes.celltype$barcode)
adata
gc()

adata@meta.data$celltypes.detailed <- barcodes.celltype$celltype[match(rownames(adata@meta.data), barcodes.celltype$barcode)]

peaks_tab <- read.table('/nfs/lab/projects/nash_nafld_liver/downstream_all/windows_peak_call/union_peaks/LiverUnionPeaks.bed', header=T)

peaks_tab$seqnames <- paste0(peaks_tab$Chr, ":", peaks_ta464444305b$Start, "-", peaks_tab$End)
peaks <- GRanges(seqnames = peaks_tab$seqnames)
peaks

# remove peaks on nonstandard chromosomes and in genomic blacklist regions
peaks_sub <- keepStandardChromosomes(peaks, pruning.mode = "coarse")
peaks_sub <- subsetByOverlaps(x = peaks_sub, ranges = blacklist_hg38_unified, invert = TRUE)
peaks_sub

# quantify counts in each peak
macs2_counts <- FeatureMatrix(
  fragments = Fragments(adata),
  features = peaks_sub[400001:527920,],
  cells = colnames(adata)
)
saveRDS(macs2_counts,'/nfs/lab/projects/nash_nafld_liver/downstream_all/windows_peak_call/Signac_GRanges_mat_union_400001:527920.RDS')

In [ ]:
# Max:            2,147,483,648
# 000001:100000 -   443,775,696
# 100001:200000 -   464,444,305
# 200001:300000 -   469,771,534
# 300001:400000 -   368,585,255
# 400001:527920 -   560,286,737

macs2_counts@p[[length(macs2_counts@p)]]

In [ ]:
sum(443775696
   ,464444305
   ,469771534
   ,368585255
   ,560286737)

In [ ]:
library(Signac)
barcodes.celltype <- read.table('/nfs/lab/projects/nash_nafld_liver/downstream_all/08052024_fnih_liver_ALL28lanes_ALL87donors_50kHVWsby16pool_HarmonyCovariates_DonorDemuxAndPoolingBatch_filtered_amulet_res10_5pctamulet_subset_clustered_>10bc_incomplete_recluster_celltypes.tsv',
            header=T, sep='\t')

In [ ]:
adata <- readRDS('/nfs/lab/projects/nash_nafld_liver/keep_demux/06052024_fnih_liver_ALL28lanes_ALL87donors_50kHVWsby16pool_HarmonyCovariates_DonorDemuxAndPoolingBatch_filtered_amulet.rds')
adata

In [ ]:
adata$RNA <- NULL
adata$SCT <- NULL
adata$RNA_raw <- NULL
gc()

In [ ]:
adata <- subset(adata, cells=barcodes.celltype$barcode)
adata
gc()

In [ ]:
adata@meta.data$celltypes.detailed <- barcodes.celltype$celltype[match(rownames(adata@meta.data), barcodes.celltype$barcode)]

In [ ]:
peaks_tab <- read.table('/nfs/lab/projects/nash_nafld_liver/downstream_all/windows_peak_call/B_summits.bed')

In [ ]:
peaks_tab$seqnames <- paste0(peaks_tab$V1, ":", peaks_tab$V2, "-", peaks_tab$V3)

In [ ]:
library(GenomicRanges)

In [ ]:
peaks <- GRanges(seqnames = peaks_tab$seqnames)
peaks

In [ ]:
library(GenomeInfoDb)
# remove peaks on nonstandard chromosomes and in genomic blacklist regions
peaks_sub <- keepStandardChromosomes(peaks, pruning.mode = "coarse")
peaks_sub <- subsetByOverlaps(x = peaks_sub, ranges = blacklist_hg38_unified, invert = TRUE)

peaks_sub

In [ ]:
# quantify counts in each peak
macs2_counts <- FeatureMatrix(
  fragments = Fragments(adata),
  features = peaks_sub,
  cells = colnames(adata)
)
saveRDS(macs2_counts,'/nfs/lab/projects/nash_nafld_liver/downstream_all/windows_peak_call/Signac_GRanges_mat.RDS')

In [ ]:
adata

In [ ]:
i <- 19

colnames(meta.1)[i]
colnames(meta.2)[i]
which(meta.1[,i] != meta.2[,i])
meta.1[which(meta.1[,i] != meta.2[,i]),1]
meta.2[which(meta.1[,i] != meta.2[,i]),1]
meta.1[which(meta.1[,i] != meta.2[,i]),i]
meta.2[which(meta.1[,i] != meta.2[,i]),i]

In [ ]:
head(meta.2)

In [ ]:
head(meta.1)[,1:19]

In [ ]:
c(
59.51, 43.00, 55.35, 55.82, 38.53, 10.76, 37.54, 12.86, 5.21,
13.78, 5.79, 6.30, 12.52, 12.29, 11.81, 6.58, 15.21,
47.66, 69.45, 11.27, 7.60, 3.02, 24.41, 9.63, 
38.53, 27.70, 13.55, 7.39, 52.80, 6.74, 13.78, 
9.61, 47.34,4.90)

In [ ]:
Juston.data <- data.frame(IL6 = c(
59.51, 43.00, 55.35, 55.82, 38.53, 10.76, 37.54, 12.86, 5.21,
13.78, 5.79, 6.30, 12.52, 12.29, 11.81, 6.58, 15.21,
47.66, 69.45, 11.27, 7.60, 3.02, 24.41, 9.63, 
38.53, 27.70, 13.55, 7.39, 52.80, 6.74, 13.78, 
9.61, 47.34,4.90),
           Immune=c(rep(x = 'Ctrl', times = 4), rep(x='Imm', times=5),
                     rep(x = 'Ctrl', times = 5), rep(x='Imm', times=4),
                     rep(x = 'Ctrl', times = 4), rep(x='Imm', times=0),
                     rep(x = 'Ctrl', times = 5), rep(x='Imm', times=1),
                     rep(x = 'Ctrl', times = 4), rep(x='Imm', times=2)),
           Diet=c(rep(x = 'Soy', times = 4), rep(x='Soy', times=5),
                     rep(x = 'CGMP', times = 5), rep(x='CGMP', times=4),
                     rep(x = 'Low', times = 4), rep(x='Low', times=0),
                     rep(x = '1:1', times = 5), rep(x='1:1', times=1),
                     rep(x = 'High', times = 4), rep(x='High', times=2)))

In [ ]:
summary(aov(IL6 ~ Immune + Diet + Immune * Diet, data = Juston.data))